In [1]:
!pip -q install pandas numpy cryptography

In [2]:
import json, glob, os
import pandas as pd

json_paths = sorted(glob.glob("/content/all_data_20250817_*.json"))
json_paths

['/content/all_data_20250817_095650.json',
 '/content/all_data_20250817_095811.json',
 '/content/all_data_20250817_095959.json',
 '/content/all_data_20250817_100018.json']

In [3]:
import json, glob, os
import pandas as pd

json_paths = sorted(glob.glob("/content/all_data_20250817_*.json"))
assert json_paths, "No all_data_*.json found in /content. Upload the exports."

def load_outputs(path):
    with open(path, "r") as f:
        data = json.load(f)
    outputs = data.get("outputs", [])
    rows = []
    for o in outputs:
        pdct = o.get("parsed_data") or {}
        q = pdct.get("quantum_sentiment")
        if q:
            rows.append({
                "source_file": os.path.basename(path),
                "script_name": o.get("script_name"),
                "timestamp": q.get("timestamp") or o.get("timestamp"),
                "cycle": q.get("cycle") or o.get("cycle"),
                "classical_host": q.get("classical_host"),
                "classical_mate": q.get("classical_mate"),
                "classical_shared": q.get("classical_shared"),
                "quantum_host": q.get("quantum_host"),
                "quantum_mate": q.get("quantum_mate"),
                "quantum_shared": q.get("quantum_shared"),
            })
    return pd.DataFrame(rows)

telemetry = pd.concat([load_outputs(p) for p in json_paths], ignore_index=True)
telemetry["timestamp"] = pd.to_datetime(telemetry["timestamp"], errors="coerce")
telemetry["cycle"] = pd.to_numeric(telemetry["cycle"], errors="coerce").astype("Int64")
telemetry = telemetry.sort_values(["timestamp","script_name","cycle"], na_position="last").reset_index(drop=True)

telemetry.head(), telemetry.shape

(                     source_file script_name                  timestamp  \
 0  all_data_20250817_095650.json       model 2025-08-17 09:56:43.222236   
 1  all_data_20250817_095650.json      cookie 2025-08-17 09:56:43.749270   
 2  all_data_20250817_095650.json      client 2025-08-17 09:56:44.239768   
 3  all_data_20250817_095650.json       model 2025-08-17 09:56:45.642759   
 4  all_data_20250817_095650.json       model 2025-08-17 09:56:45.643461   
 
    cycle  classical_host  classical_mate  classical_shared  quantum_host  \
 0      1           0.714           0.733             0.709         0.376   
 1      1           0.562           0.710             0.603         0.272   
 2      1           0.468           0.714             0.349         0.288   
 3      1           0.714           0.733             0.709         0.376   
 4      2           0.714           0.733             0.709         0.376   
 
    quantum_mate  quantum_shared  
 0         0.328           0.240  
 1      

In [4]:
import numpy as np
from cryptography.hazmat.primitives.kdf.hkdf import HKDF
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.backends import default_backend

def key_from_row(row, salt=b"hive-v1", info=b"sentiment-key"):
    # robust quantization: map floats -> int16 bytes deterministically
    vec = np.array([row["quantum_host"], row["quantum_mate"], row["quantum_shared"]], dtype=np.float64)
    if np.any(pd.isna(vec)):
        return None
    # clip to [-1,1] then scale
    vec = np.clip(vec, -1.0, 1.0)
    q = (vec * 32767.0).round().astype(np.int16)
    ikm = q.tobytes() + str(row.get("cycle")).encode()  # include cycle for uniqueness

    hkdf = HKDF(
        algorithm=hashes.SHA256(),
        length=32,
        salt=salt,
        info=info,
        backend=default_backend()
    )
    return hkdf.derive(ikm)

# attach keys for rows that have quantum values
telemetry["key32"] = telemetry.apply(key_from_row, axis=1)
telemetry[telemetry["key32"].notna()].head(5)


,source_file,script_name,timestamp,cycle,classical_host,classical_mate,classical_shared,quantum_host,quantum_mate,quantum_shared,key32
0,all_data_20250817_095650.json,model,2025-08-17 09:56:43.222236,1,0.714,0.733,0.709,0.376,0.328,0.240,b'\x1cD\xd2\xb5\x9c\xf2M\xaa\xd6\xe3\x18uu\xc5...
1,all_data_20250817_095650.json,cookie,2025-08-17 09:56:43.749270,1,0.562,0.710,0.603,0.272,0.288,0.216,b' B~Fk\xbb\xe5\xafo=\xb8\x1e\xb9\x04ipI0\xc1}...
2,all_data_20250817_095650.json,client,2025-08-17 09:56:44.239768,1,0.468,0.714,0.349,0.288,0.312,0.224,b'\xf0\xe5k\x8c\xc3\xf2\x9c\xef\xe5\xdf\x82\x0...
3,all_data_20250817_095650.json,model,2025-08-17 09:56:45.642759,1,0.714,0.733,0.709,0.376,0.328,0.240,b'\x1cD\xd2\xb5\x9c\xf2M\xaa\xd6\xe3\x18uu\xc5...
4,all_data_20250817_095650.json,model,2025-08-17 09:56:45.643461,2,0.714,0.733,0.709,0.376,0.328,0.240,b'_D\x90\xba3\x9e>\x04\xd3\xdfJ\xfc\xe7\x9b\x1...


In [5]:
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
import os, base64

def encrypt_with_row(row, plaintext: bytes, aad: bytes = b"hive"):
    key = row["key32"]
    if key is None:
        raise ValueError("Row has no key")
    aesgcm = AESGCM(key)
    nonce = os.urandom(12)
    ct = aesgcm.encrypt(nonce, plaintext, aad)
    return {
        "nonce_b64": base64.b64encode(nonce).decode(),
        "ct_b64": base64.b64encode(ct).decode(),
        "cycle": int(row["cycle"]) if pd.notna(row["cycle"]) else None,
        "timestamp": str(row["timestamp"]),
        "script_name": row["script_name"],
    }

def decrypt_with_row(row, payload, aad: bytes = b"hive"):
    key = row["key32"]
    aesgcm = AESGCM(key)
    nonce = base64.b64decode(payload["nonce_b64"])
    ct = base64.b64decode(payload["ct_b64"])
    return aesgcm.decrypt(nonce, ct, aad)

# pick a row that has a key
row = telemetry[telemetry["key32"].notna()].iloc[0]
payload = encrypt_with_row(row, b"hello hive: quantum-locked message")
payload


{'nonce_b64': '/Kyyhcz30B9vCG/4',
 'ct_b64': 'iwwJGrF/G5qhZTMjDkCpu7D4hjTNDFNtnmQzPHjyp38ysnhnGQPqRxRvPAwjpVPf1jE=',
 'cycle': 1,
 'timestamp': '2025-08-17 09:56:43.222236',
 'script_name': 'model'}

In [6]:
import sqlite3, hashlib, json
from pathlib import Path

db_path = "/content/collected_data.db"
assert Path(db_path).exists(), "Upload collected_data.db into /content first"

con = sqlite3.connect(db_path)
cur = con.cursor()

cur.execute("""
CREATE TABLE IF NOT EXISTS hive_telemetry (
  id INTEGER PRIMARY KEY AUTOINCREMENT,
  timestamp TEXT,
  cycle INTEGER,
  script_name TEXT,
  source_file TEXT,
  classical_host REAL,
  classical_mate REAL,
  classical_shared REAL,
  quantum_host REAL,
  quantum_mate REAL,
  quantum_shared REAL,
  key_sha256 TEXT
)
""")

cur.execute("""
CREATE TABLE IF NOT EXISTS hive_messages (
  id INTEGER PRIMARY KEY AUTOINCREMENT,
  created_at TEXT,
  cycle INTEGER,
  script_name TEXT,
  aad TEXT,
  nonce_b64 TEXT,
  ct_b64 TEXT,
  key_sha256 TEXT,
  meta_json TEXT
)
""")

cur.execute("CREATE INDEX IF NOT EXISTS idx_tel_cycle ON hive_telemetry(cycle)")
cur.execute("CREATE INDEX IF NOT EXISTS idx_msg_cycle ON hive_messages(cycle)")
con.commit()

print("DB ready:", db_path)


DB ready: /content/collected_data.db


In [7]:
import pandas as pd
import hashlib # Added missing import

def sha256_hex(b: bytes) -> str:
    return hashlib.sha256(b).hexdigest()

# Ensure the database schema is up-to-date for this cell's operation
# Drop the table if it exists to allow schema re-creation with the key_sha256 column
cur.execute("DROP TABLE IF EXISTS hive_telemetry")
cur.execute("""
CREATE TABLE IF NOT EXISTS hive_telemetry (
  id INTEGER PRIMARY KEY AUTOINCREMENT,
  timestamp TEXT,
  cycle INTEGER,
  script_name TEXT,
  source_file TEXT,
  classical_host REAL,
  classical_mate REAL,
  classical_shared REAL,
  quantum_host REAL,
  quantum_mate REAL,
  quantum_shared REAL,
  key_sha256 TEXT
)
""")
con.commit()

tel = telemetry.copy()
tel = tel[tel["key32"].notna()].copy()
tel["key_sha256"] = tel["key32"].apply(lambda k: sha256_hex(k))

# write to db (append)
cols = [
    "timestamp","cycle","script_name","source_file",
    "classical_host","classical_mate","classical_shared",
    "quantum_host","quantum_mate","quantum_shared",
    "key_sha256"
]
tel_to_write = tel[cols].copy()
tel_to_write["timestamp"] = tel_to_write["timestamp"].astype(str)

tel_to_write.to_sql("hive_telemetry", con, if_exists="append", index=False)

con.commit()
print("Inserted telemetry rows:", len(tel_to_write))

Inserted telemetry rows: 49


In [8]:
pd.read_sql_query("SELECT cycle, script_name, timestamp, key_sha256 FROM hive_telemetry ORDER BY id DESC LIMIT 10", con)


,cycle,script_name,timestamp,key_sha256
0,1,blockheart,2025-08-17 10:00:17.212964,f9d1ecf7b4dd60af071665f49a5768740960141d7a665b...
1,1,blockheart,2025-08-17 10:00:17.127296,f9d1ecf7b4dd60af071665f49a5768740960141d7a665b...
2,1,model,2025-08-17 10:00:16.397031,ca1a6f12aad5c1e27eb6e48b332f6e0b58eaa5026d4c36...
3,1,brian,2025-08-17 10:00:15.636896,e6fbbb78e882abc6fb1e3d5c76dcc5c25ab9d007663439...
4,4,cookie,2025-08-17 09:56:49.464948,ede290ab19e953a870f7f026c3b0619ca6d6772438132c...
5,4,cookie,2025-08-17 09:56:49.464790,ede290ab19e953a870f7f026c3b0619ca6d6772438132c...
6,4,cookie,2025-08-17 09:56:49.384530,ede290ab19e953a870f7f026c3b0619ca6d6772438132c...
7,4,cookie,2025-08-17 09:56:49.384412,ecbd04bcdc429120a2133726b1017082597920a3454627...
8,4,cookie,2025-08-17 09:56:49.384299,ecbd04bcdc429120a2133726b1017082597920a3454627...
9,4,cookie,2025-08-17 09:56:49.384174,ecbd04bcdc429120a2133726b1017082597920a3454627...


In [9]:
from datetime import datetime
import base64

def store_message(row, plaintext: bytes, aad: bytes = b"hive"):
    payload = encrypt_with_row(row, plaintext, aad=aad)

    key_sha = sha256_hex(row["key32"])
    meta = {
        "source_file": row.get("source_file"),
        "ts": str(row.get("timestamp")),
        "classical": {
            "host": float(row.get("classical_host")),
            "mate": float(row.get("classical_mate")),
            "shared": float(row.get("classical_shared")),
        },
        "quantum": {
            "host": float(row.get("quantum_host")),
            "mate": float(row.get("quantum_mate")),
            "shared": float(row.get("quantum_shared")),
        }
    }

    cur.execute("""
      INSERT INTO hive_messages (created_at, cycle, script_name, aad, nonce_b64, ct_b64, key_sha256, meta_json)
      VALUES (?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        datetime.utcnow().isoformat(timespec="seconds") + "Z",
        int(row["cycle"]) if pd.notna(row["cycle"]) else None,
        str(row["script_name"]),
        aad.decode("utf-8", errors="replace"),
        payload["nonce_b64"],
        payload["ct_b64"],
        key_sha,
        json.dumps(meta, ensure_ascii=False),
    ))
    con.commit()
    return payload

# pick a row (you can choose a specific cycle later)
row = telemetry[telemetry["key32"].notna()].iloc[0]
payload = store_message(row, b"hello hive: stored in sqlite")
payload


/tmp/ipython-input-2367696085.py:27: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  datetime.utcnow().isoformat(timespec="seconds") + "Z",


{'nonce_b64': 'LaC5q3+4IaQGIPk9',
 'ct_b64': 'S9xQcTE5u+RnPdMI0BCHkVEFobWEYBnnoISmvDOpKRDFvG1mKzs141kgRK0=',
 'cycle': 1,
 'timestamp': '2025-08-17 09:56:43.222236',
 'script_name': 'model'}

In [10]:
pd.read_sql_query("SELECT id, created_at, cycle, script_name, aad, key_sha256 FROM hive_messages ORDER BY id DESC LIMIT 5", con)

,id,created_at,cycle,script_name,aad,key_sha256
0,1,2026-01-19T18:29:58Z,1,model,hive,34a65f0ff239b21a299084769984c3d06cf3f6b068b443...


In [11]:
def load_latest_message():
    df = pd.read_sql_query("SELECT * FROM hive_messages ORDER BY id DESC LIMIT 1", con)
    return df.iloc[0].to_dict()

msg = load_latest_message()
msg


{'id': 1,
 'created_at': '2026-01-19T18:29:58Z',
 'cycle': 1,
 'script_name': 'model',
 'aad': 'hive',
 'nonce_b64': 'LaC5q3+4IaQGIPk9',
 'ct_b64': 'S9xQcTE5u+RnPdMI0BCHkVEFobWEYBnnoISmvDOpKRDFvG1mKzs141kgRK0=',
 'key_sha256': '34a65f0ff239b21a299084769984c3d06cf3f6b068b4437254b3c3375970b67c',
 'meta_json': '{"source_file": "all_data_20250817_095650.json", "ts": "2025-08-17 09:56:43.222236", "classical": {"host": 0.714, "mate": 0.733, "shared": 0.709}, "quantum": {"host": 0.376, "mate": 0.328, "shared": 0.24}}'}

In [12]:
cycle = msg["cycle"]
script = msg["script_name"]

candidate = telemetry[
    (telemetry["cycle"] == cycle) &
    (telemetry["script_name"] == script) &
    (telemetry["key32"].notna())
].iloc[0]

pt = decrypt_with_row(candidate, msg, aad=msg["aad"].encode())
pt


b'hello hive: stored in sqlite'

In [13]:
!pip -q install cirq


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 670.8/670.8 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.5/73.5 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 430.5/430.5 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 52.6 MB/s eta 0:00:00


In [14]:
import cirq
import numpy as np

def circuit_projection_bits(row, n_qubits=6, reps=256):
    # map floats [-1,1] -> angles
    v = np.array([row["quantum_host"], row["quantum_mate"], row["quantum_shared"]], dtype=float)
    v = np.clip(v, -1.0, 1.0)
    angles = (v + 1.0) * np.pi  # [0, 2π]

    qs = cirq.LineQubit.range(n_qubits)
    c = cirq.Circuit()

    # simple “sentiment embedding” across qubits
    for i,q in enumerate(qs):
        a = angles[i % 3]
        c.append(cirq.ry(a)(q))
        c.append(cirq.rz(a/2)(q))

    # light entanglement
    for i in range(n_qubits-1):
        c.append(cirq.CNOT(qs[i], qs[i+1]))

    c.append(cirq.measure(*qs, key="m"))

    sim = cirq.Simulator()
    res = sim.run(c, repetitions=reps)
    bits = res.measurements["m"]  # shape (reps, n_qubits)
    # compress to bytes deterministically
    packed = np.packbits(bits.astype(np.uint8), axis=1).tobytes()
    return packed  # bytes

def key_from_row_cirq(row, salt=b"hive-v1", info=b"cirq-proj"):
    ikm = circuit_projection_bits(row) + str(row.get("cycle")).encode()
    hkdf = HKDF(
        algorithm=hashes.SHA256(),
        length=32,
        salt=salt,
        info=info,
        backend=default_backend()
    )
    return hkdf.derive(ikm)

# Example: generate cirq-based key
row = telemetry[telemetry["key32"].notna()].iloc[0]
k_cirq = key_from_row_cirq(row)
hashlib.sha256(k_cirq).hexdigest()[:16]


'17fdb96e10933eef'

In [15]:
!pip -q install pandas numpy cryptography scikit-learn cirq


In [16]:
import json, glob, os
import pandas as pd

json_paths = sorted(glob.glob("/content/all_data_20250817_*.json"))
assert json_paths, "No all_data_*.json found in /content. Upload the exports."

def load_outputs(path):
    with open(path, "r") as f:
        data = json.load(f)
    outputs = data.get("outputs", [])
    rows = []
    for o in outputs:
        pdct = o.get("parsed_data") or {}
        q = pdct.get("quantum_sentiment")
        if q:
            rows.append({
                "source_file": os.path.basename(path),
                "script_name": o.get("script_name"),
                "timestamp": q.get("timestamp") or o.get("timestamp"),
                "cycle": q.get("cycle") or o.get("cycle"),
                "classical_host": q.get("classical_host"),
                "classical_mate": q.get("classical_mate"),
                "classical_shared": q.get("classical_shared"),
                "quantum_host": q.get("quantum_host"),
                "quantum_mate": q.get("quantum_mate"),
                "quantum_shared": q.get("quantum_shared"),
            })
    return pd.DataFrame(rows)

telemetry = pd.concat([load_outputs(p) for p in json_paths], ignore_index=True)
telemetry["timestamp"] = pd.to_datetime(telemetry["timestamp"], errors="coerce")
telemetry["cycle"] = pd.to_numeric(telemetry["cycle"], errors="coerce").astype("Int64")
telemetry = telemetry.sort_values(["timestamp","script_name","cycle"], na_position="last").reset_index(drop=True)

telemetry.head(), telemetry.shape


(                     source_file script_name                  timestamp  \
 0  all_data_20250817_095650.json       model 2025-08-17 09:56:43.222236   
 1  all_data_20250817_095650.json      cookie 2025-08-17 09:56:43.749270   
 2  all_data_20250817_095650.json      client 2025-08-17 09:56:44.239768   
 3  all_data_20250817_095650.json       model 2025-08-17 09:56:45.642759   
 4  all_data_20250817_095650.json       model 2025-08-17 09:56:45.643461   
 
    cycle  classical_host  classical_mate  classical_shared  quantum_host  \
 0      1           0.714           0.733             0.709         0.376   
 1      1           0.562           0.710             0.603         0.272   
 2      1           0.468           0.714             0.349         0.288   
 3      1           0.714           0.733             0.709         0.376   
 4      2           0.714           0.733             0.709         0.376   
 
    quantum_mate  quantum_shared  
 0         0.328           0.240  
 1      

In [17]:
import numpy as np
import hashlib
from cryptography.hazmat.primitives.kdf.hkdf import HKDF
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.backends import default_backend
import cirq

def sha256_hex(b: bytes) -> str:
    return hashlib.sha256(b).hexdigest()

def hkdf32(ikm: bytes, salt=b"hive-v1", info=b"sentiment-key") -> bytes:
    hkdf = HKDF(
        algorithm=hashes.SHA256(),
        length=32,
        salt=salt,
        info=info,
        backend=default_backend()
    )
    return hkdf.derive(ikm)

def key_from_row_simple(row, salt=b"hive-v1", info=b"simple-v1"):
    vec = np.array([row["quantum_host"], row["quantum_mate"], row["quantum_shared"]], dtype=np.float64)
    if np.any(pd.isna(vec)):
        return None
    vec = np.clip(vec, -1.0, 1.0)
    q = (vec * 32767.0).round().astype(np.int16)
    ikm = q.tobytes() + str(int(row["cycle"]) if pd.notna(row["cycle"]) else -1).encode()
    return hkdf32(ikm, salt=salt, info=info)

def circuit_projection_bytes(row, n_qubits=6, reps=256):
    v = np.array([row["quantum_host"], row["quantum_mate"], row["quantum_shared"]], dtype=float)
    if np.any(pd.isna(v)):
        return None
    v = np.clip(v, -1.0, 1.0)
    angles = (v + 1.0) * np.pi  # [0, 2π]

    qs = cirq.LineQubit.range(n_qubits)
    c = cirq.Circuit()

    for i,qb in enumerate(qs):
        a = angles[i % 3]
        c.append(cirq.ry(a)(qb))
        c.append(cirq.rz(a/2)(qb))
    for i in range(n_qubits-1):
        c.append(cirq.CNOT(qs[i], qs[i+1]))

    c.append(cirq.measure(*qs, key="m"))

    sim = cirq.Simulator()
    res = sim.run(c, repetitions=reps)
    bits = res.measurements["m"].astype(np.uint8)  # (reps, n_qubits)
    packed = np.packbits(bits, axis=1).tobytes()
    return packed

def key_from_row_cirq(row, salt=b"hive-v1", info=b"cirq-proj-v1"):
    packed = circuit_projection_bytes(row)
    if packed is None:
        return None
    ikm = packed + str(int(row["cycle"]) if pd.notna(row["cycle"]) else -1).encode()
    return hkdf32(ikm, salt=salt, info=info)

telemetry["key32_simple"] = telemetry.apply(key_from_row_simple, axis=1)
telemetry["key32_cirq"]   = telemetry.apply(key_from_row_cirq, axis=1)

telemetry["key_sha_simple"] = telemetry["key32_simple"].apply(lambda k: sha256_hex(k) if isinstance(k,(bytes,bytearray)) else None)
telemetry["key_sha_cirq"]   = telemetry["key32_cirq"].apply(lambda k: sha256_hex(k) if isinstance(k,(bytes,bytearray)) else None)

telemetry[telemetry["key_sha_cirq"].notna()].head()


,source_file,script_name,timestamp,cycle,classical_host,classical_mate,classical_shared,quantum_host,quantum_mate,quantum_shared,key32_simple,key32_cirq,key_sha_simple,key_sha_cirq
0,all_data_20250817_095650.json,model,2025-08-17 09:56:43.222236,1,0.714,0.733,0.709,0.376,0.328,0.240,b'\xb93\x9a\x08\x91\x1d\x13\xb64\')\xf1\x18\xf...,b'\xfev4\xb91\x14\xef\xc7r\x96\xb3a\xa4\x0b\x9...,4bc5209475d2bfdd7c11f99841e59755ed1624bbcd37e6...,84bacf7270671294e746ae76557d9f131249aaab9805bc...
1,all_data_20250817_095650.json,cookie,2025-08-17 09:56:43.749270,1,0.562,0.710,0.603,0.272,0.288,0.216,b'\xb6\x07.g\xa4\xf9L\xdb\x1f:\x04\x114\xe5\x1...,"b'""\xa7\xf5_g\x872\x1b<\xe95\x188\xad\xf8=9E+j...",10e20666fd8fd70b1d1bd7baf01ecc3cdf86b3162b4edb...,503d486d73e3d7bd571eab43582dcd81dce615a8f58d96...
2,all_data_20250817_095650.json,client,2025-08-17 09:56:44.239768,1,0.468,0.714,0.349,0.288,0.312,0.224,b'~\xaaG=\rUU\t\x8e\x057_\xc3\xe2\xebNQd3t\xfc...,b'.S\xb9j\x94Mh\xb9\xc8\x0e_\xd5=^\x02\xa9\x85...,dd201754f43ceff13d37ac555b7f55838320978487b434...,d5b65d29c1ab43acec8958b3fe7d1d9a0babdef533841f...
3,all_data_20250817_095650.json,model,2025-08-17 09:56:45.642759,1,0.714,0.733,0.709,0.376,0.328,0.240,b'\xb93\x9a\x08\x91\x1d\x13\xb64\')\xf1\x18\xf...,b'>E\x8f\xe09\x9a\\\x06\xf0m\xf57\xd4\xcd\xa8\...,4bc5209475d2bfdd7c11f99841e59755ed1624bbcd37e6...,d0d6221fd4b5abdd83efbbc9f1ba7564e0ed188c5a1504...
4,all_data_20250817_095650.json,model,2025-08-17 09:56:45.643461,2,0.714,0.733,0.709,0.376,0.328,0.240,b'`d\xaa\xb0\xb0\x9d\xa5\xb0\xcd\xf3sjI\xc6r\x...,b'\xd2\xf0\xeer\xb9\x08\x9e[uk$\n\x06a}\x9e\x0...,b66d4d66274f41dced77dd092582550091072aeb3ee84c...,5819daf567a6f1dc63b4e4f6eba66a77d9fe45bcc76d95...


In [18]:
import sqlite3
from pathlib import Path

db_path = "/content/collected_data.db"
assert Path(db_path).exists(), "Upload collected_data.db into /content first"

con = sqlite3.connect(db_path)
cur = con.cursor()

# Drop the tables if they exist to ensure schema update
cur.execute("DROP TABLE IF EXISTS hive_telemetry")
cur.execute("DROP TABLE IF EXISTS hive_messages")

cur.execute("""
CREATE TABLE IF NOT EXISTS hive_telemetry (
  timestamp TEXT NOT NULL,
  cycle INTEGER,
  script_name TEXT NOT NULL,
  source_file TEXT,
  classical_host REAL,
  classical_mate REAL,
  classical_shared REAL,
  quantum_host REAL,
  quantum_mate REAL,
  quantum_shared REAL,
  key_sha_simple TEXT,
  key_sha_cirq TEXT,
  PRIMARY KEY (timestamp, script_name, cycle)
)
""")

cur.execute("""
CREATE TABLE IF NOT EXISTS hive_messages (
  id INTEGER PRIMARY KEY AUTOINCREMENT,
  created_at TEXT NOT NULL,
  timestamp TEXT,
  cycle INTEGER,
  script_name TEXT,
  aad TEXT,
  nonce_b64 TEXT,
  ct_b64 TEXT,
  key_sha TEXT,
  key_kind TEXT,
  meta_json TEXT
)
""")

cur.execute("CREATE INDEX IF NOT EXISTS idx_tel_cycle ON hive_telemetry(cycle)")
cur.execute("CREATE INDEX IF NOT EXISTS idx_msg_cycle ON hive_messages(cycle)")
con.commit()

print("DB schema ready.")

DB schema ready.


In [19]:
tel = telemetry.copy()
tel = tel[tel["timestamp"].notna() & tel["script_name"].notna()].copy()

def to_py(v):
    if pd.isna(v): return None
    if isinstance(v, (pd.Timestamp,)): return v.isoformat()
    if isinstance(v, (pd._libs.missing.NAType,)): return None
    return v

rows = []
for _, r in tel.iterrows():
    rows.append((
        to_py(r["timestamp"]),
        int(r["cycle"]) if pd.notna(r["cycle"]) else None,
        str(r["script_name"]),
        str(r["source_file"]) if pd.notna(r["source_file"]) else None,
        to_py(r["classical_host"]), to_py(r["classical_mate"]), to_py(r["classical_shared"]),
        to_py(r["quantum_host"]),   to_py(r["quantum_mate"]),   to_py(r["quantum_shared"]),
        r["key_sha_simple"],
        r["key_sha_cirq"],
    ))

cur.executemany("""
INSERT INTO hive_telemetry (
  timestamp, cycle, script_name, source_file,
  classical_host, classical_mate, classical_shared,
  quantum_host, quantum_mate, quantum_shared,
  key_sha_simple, key_sha_cirq
) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
ON CONFLICT(timestamp, script_name, cycle) DO UPDATE SET
  source_file=excluded.source_file,
  classical_host=excluded.classical_host,
  classical_mate=excluded.classical_mate,
  classical_shared=excluded.classical_shared,
  quantum_host=excluded.quantum_host,
  quantum_mate=excluded.quantum_mate,
  quantum_shared=excluded.quantum_shared,
  key_sha_simple=excluded.key_sha_simple,
  key_sha_cirq=excluded.key_sha_cirq
""", rows)

con.commit()
print("Upserted telemetry rows:", len(rows))

Upserted telemetry rows: 49


In [20]:
import os, base64, json
from datetime import datetime
from cryptography.hazmat.primitives.ciphers.aead import AESGCM

def encrypt_with_key(key32: bytes, plaintext: bytes, aad: bytes=b"hive"):
    aesgcm = AESGCM(key32)
    nonce = os.urandom(12)
    ct = aesgcm.encrypt(nonce, plaintext, aad)
    return base64.b64encode(nonce).decode(), base64.b64encode(ct).decode()

def decrypt_with_key(key32: bytes, nonce_b64: str, ct_b64: str, aad: bytes=b"hive"):
    aesgcm = AESGCM(key32)
    nonce = base64.b64decode(nonce_b64)
    ct = base64.b64decode(ct_b64)
    return aesgcm.decrypt(nonce, ct, aad)

def store_message(row, plaintext: bytes, key_kind="cirq", aad: bytes=b"hive"):
    key32 = row["key32_cirq"] if key_kind == "cirq" else row["key32_simple"]
    if not isinstance(key32, (bytes, bytearray)):
        raise ValueError("Row has no key for kind=" + key_kind)

    nonce_b64, ct_b64 = encrypt_with_key(key32, plaintext, aad=aad)
    key_sha = sha256_hex(key32)

    meta = {
        "source_file": row.get("source_file"),
        "ts": str(row.get("timestamp")),
        "classical": {
            "host": float(row.get("classical_host")) if pd.notna(row.get("classical_host")) else None,
            "mate": float(row.get("classical_mate")) if pd.notna(row.get("classical_mate")) else None,
            "shared": float(row.get("classical_shared")) if pd.notna(row.get("classical_shared")) else None,
        },
        "quantum": {
            "host": float(row.get("quantum_host")) if pd.notna(row.get("quantum_host")) else None,
            "mate": float(row.get("quantum_mate")) if pd.notna(row.get("quantum_mate")) else None,
            "shared": float(row.get("quantum_shared")) if pd.notna(row.get("quantum_shared")) else None,
        }
    }

    cur.execute("""
      INSERT INTO hive_messages (created_at, timestamp, cycle, script_name, aad, nonce_b64, ct_b64, key_sha, key_kind, meta_json)
      VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        datetime.utcnow().isoformat(timespec="seconds") + "Z",
        str(row["timestamp"]),
        int(row["cycle"]) if pd.notna(row["cycle"]) else None,
        str(row["script_name"]),
        aad.decode("utf-8", errors="replace"),
        nonce_b64, ct_b64,
        key_sha, key_kind,
        json.dumps(meta, ensure_ascii=False),
    ))
    con.commit()
    return nonce_b64, ct_b64, key_sha

row0 = telemetry[telemetry["key32_cirq"].notna()].iloc[0]
nonce_b64, ct_b64, key_sha = store_message(row0, b"hello hive: cirq-locked message", key_kind="cirq")
(key_sha, nonce_b64[:10], ct_b64[:10])

/tmp/ipython-input-2248142252.py:44: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  datetime.utcnow().isoformat(timespec="seconds") + "Z",


('84bacf7270671294e746ae76557d9f131249aaab9805bcac4471b2f8b3cc2c88',
 'scJdB3m5gD',
 'YoYKug0+lF')

In [21]:
import os, base64, json
from datetime import datetime
from cryptography.hazmat.primitives.ciphers.aead import AESGCM

def encrypt_with_key(key32: bytes, plaintext: bytes, aad: bytes=b"hive"):
    aesgcm = AESGCM(key32)
    nonce = os.urandom(12)
    ct = aesgcm.encrypt(nonce, plaintext, aad)
    return base64.b64encode(nonce).decode(), base64.b64encode(ct).decode()

def decrypt_with_key(key32: bytes, nonce_b64: str, ct_b64: str, aad: bytes=b"hive"):
    aesgcm = AESGCM(key32)
    nonce = base64.b64decode(nonce_b64)
    ct = base64.b64decode(ct_b64)
    return aesgcm.decrypt(nonce, ct, aad)

def store_message(row, plaintext: bytes, key_kind="cirq", aad: bytes=b"hive"):
    key32 = row["key32_cirq"] if key_kind == "cirq" else row["key32_simple"]
    if not isinstance(key32, (bytes, bytearray)):
        raise ValueError("Row has no key for kind=" + key_kind)

    nonce_b64, ct_b64 = encrypt_with_key(key32, plaintext, aad=aad)
    key_sha = sha256_hex(key32)

    meta = {
        "source_file": row.get("source_file"),
        "ts": str(row.get("timestamp")),
        "classical": {
            "host": float(row.get("classical_host")) if pd.notna(row.get("classical_host")) else None,
            "mate": float(row.get("classical_mate")) if pd.notna(row.get("classical_mate")) else None,
            "shared": float(row.get("classical_shared")) if pd.notna(row.get("classical_shared")) else None,
        },
        "quantum": {
            "host": float(row.get("quantum_host")) if pd.notna(row.get("quantum_host")) else None,
            "mate": float(row.get("quantum_mate")) if pd.notna(row.get("quantum_mate")) else None,
            "shared": float(row.get("quantum_shared")) if pd.notna(row.get("quantum_shared")) else None,
        }
    }

    cur.execute("""
      INSERT INTO hive_messages (created_at, timestamp, cycle, script_name, aad, nonce_b64, ct_b64, key_sha, key_kind, meta_json)
      VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        datetime.utcnow().isoformat(timespec="seconds") + "Z",
        str(row["timestamp"]),
        int(row["cycle"]) if pd.notna(row["cycle"]) else None,
        str(row["script_name"]),
        aad.decode("utf-8", errors="replace"),
        nonce_b64, ct_b64,
        key_sha, key_kind,
        json.dumps(meta, ensure_ascii=False),
    ))
    con.commit()
    return nonce_b64, ct_b64, key_sha

row0 = telemetry[telemetry["key32_cirq"].notna()].iloc[0]
nonce_b64, ct_b64, key_sha = store_message(row0, b"hello hive: cirq-locked message", key_kind="cirq")
(key_sha, nonce_b64[:10], ct_b64[:10])


/tmp/ipython-input-330299270.py:44: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  datetime.utcnow().isoformat(timespec="seconds") + "Z",


('84bacf7270671294e746ae76557d9f131249aaab9805bcac4471b2f8b3cc2c88',
 'CNV31/M6+y',
 'ZmfFh7V8wb')

In [22]:
import numpy as np

df = telemetry.copy()
df = df[df["timestamp"].notna()].copy()

# Basic features
df["hour"] = df["timestamp"].dt.hour.astype(float)
df["minute"] = df["timestamp"].dt.minute.astype(float)

# Target(s)
targets = ["quantum_host","quantum_mate","quantum_shared"]

# Keep rows where we have classical + quantum
needed = ["classical_host","classical_mate","classical_shared"] + targets
df = df.dropna(subset=needed + ["script_name"])

# One-hot encode script_name
X = df[["classical_host","classical_mate","classical_shared","hour","minute"]].copy()
X = pd.concat([X, pd.get_dummies(df["script_name"], prefix="script")], axis=1)

y_shared = df["quantum_shared"].astype(float).values
y_host   = df["quantum_host"].astype(float).values
y_mate   = df["quantum_mate"].astype(float).values

X.shape, df.shape


((49, 10), (49, 16))

In [23]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor

X_train, X_test, y_train, y_test = train_test_split(X, y_shared, test_size=0.2, random_state=42)

reg = RandomForestRegressor(
    n_estimators=400,
    random_state=42,
    n_jobs=-1
)
reg.fit(X_train, y_train)
pred = reg.predict(X_test)

print("MAE:", mean_absolute_error(y_test, pred))
print("R2 :", r2_score(y_test, pred))


MAE: 0.010541086652236511
R2 : -0.6333787491091676


In [24]:
from sklearn.ensemble import IsolationForest

feat_cols = ["classical_host","classical_mate","classical_shared","quantum_host","quantum_mate","quantum_shared"]
A = df[feat_cols].astype(float).values

iso = IsolationForest(n_estimators=400, contamination=0.03, random_state=42)
scores = iso.fit_predict(A)  # -1 anomaly, +1 normal
df["anomaly"] = (scores == -1)

df[df["anomaly"]].head(20)[["timestamp","cycle","script_name"] + feat_cols]


,timestamp,cycle,script_name,classical_host,classical_mate,classical_shared,quantum_host,quantum_mate,quantum_shared
32,2025-08-17 09:56:48.403926,3,client,0.786,0.485,0.733,0.456,0.344,0.344
36,2025-08-17 09:56:49.019539,4,model,0.633,0.588,0.293,0.424,0.416,0.208


In [25]:
df.groupby("script_name")["anomaly"].mean().sort_values(ascending=False)


,anomaly
script_name,
client,0.090909
model,0.058824
blockheart,0.000000
brian,0.000000
cookie,0.000000


In [26]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split

y_script = df["script_name"].values
Xc = df[feat_cols + ["hour","minute"]].astype(float)
Xc = (Xc - Xc.mean()) / (Xc.std() + 1e-9)  # standardize

# Check the distribution of classes in y_script
print("Script name counts:\n", df["script_name"].value_counts())

# Filter out script names with only one occurrence
script_counts = df["script_name"].value_counts()
scripts_to_keep = script_counts[script_counts > 1].index
df_filtered = df[df["script_name"].isin(scripts_to_keep)]

y_script_filtered = df_filtered["script_name"].values
Xc_filtered = Xc[df["script_name"].isin(scripts_to_keep)]

X_train, X_test, y_train, y_test = train_test_split(Xc_filtered, y_script_filtered, test_size=0.2, random_state=42, stratify=y_script_filtered)

clf = LogisticRegression(max_iter=2000)
clf.fit(X_train, y_train)
pred = clf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred))

Script name counts:
 script_name
cookie        18
model         17
client        11
blockheart     2
brian          1
Name: count, dtype: int64
Accuracy: 0.8
              precision    recall  f1-score   support

      client       0.00      0.00      0.00         2
      cookie       0.67      1.00      0.80         4
       model       1.00      1.00      1.00         4

    accuracy                           0.80        10
   macro avg       0.56      0.67      0.60        10
weighted avg       0.67      0.80      0.72        10



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [27]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split

y_script = df["script_name"].values
Xc = df[feat_cols + ["hour","minute"]].astype(float)
Xc = (Xc - Xc.mean()) / (Xc.std() + 1e-9)  # standardize

# Filter out script names with only one occurrence to allow for stratified splitting
script_counts = df["script_name"].value_counts()
scripts_to_keep = script_counts[script_counts > 1].index
df_filtered = df[df["script_name"].isin(scripts_to_keep)]

y_script_filtered = df_filtered["script_name"].values
Xc_filtered = Xc[df["script_name"].isin(scripts_to_keep)]

X_train, X_test, y_train, y_test = train_test_split(Xc_filtered, y_script_filtered, test_size=0.2, random_state=42, stratify=y_script_filtered)

clf = LogisticRegression(max_iter=2000)
clf.fit(X_train, y_train)
pred = clf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred))

Accuracy: 0.8
              precision    recall  f1-score   support

      client       0.00      0.00      0.00         2
      cookie       0.67      1.00      0.80         4
       model       1.00      1.00      1.00         4

    accuracy                           0.80        10
   macro avg       0.56      0.67      0.60        10
weighted avg       0.67      0.80      0.72        10



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [28]:
print("rows:", len(telemetry))
print("keys(simple):", telemetry["key32_simple"].notna().sum())
print("keys(cirq):", telemetry["key32_cirq"].notna().sum())
print("unique scripts:", telemetry["script_name"].nunique())
print("scripts:", sorted(telemetry["script_name"].dropna().unique())[:20])


rows: 49
keys(simple): 49
keys(cirq): 49
unique scripts: 5
scripts: ['blockheart', 'brian', 'client', 'cookie', 'model']


In [29]:
!pip -q install pandas numpy scikit-learn torch torchvision torchaudio

# Cirq + qsimcirq can be finicky; this usually works in Colab:
!pip -q install cirq qsimcirq

# Brian2
!pip -q install brian2


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 539.3/539.3 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 293.6/293.6 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 28.9 MB/s eta 0:00:00


In [30]:
import json, glob, os
import pandas as pd
import numpy as np

json_paths = sorted(glob.glob("/content/all_data_20250817_*.json"))
assert json_paths, "Upload all_data_20250817_*.json files to /content."

def load_outputs(path):
    with open(path, "r") as f:
        data = json.load(f)
    outputs = data.get("outputs", [])
    rows = []
    for o in outputs:
        pdct = o.get("parsed_data") or {}
        q = pdct.get("quantum_sentiment")
        if q:
            rows.append({
                "source_file": os.path.basename(path),
                "script_name": o.get("script_name"),
                "timestamp": q.get("timestamp") or o.get("timestamp"),
                "cycle": q.get("cycle") or o.get("cycle"),
                "classical_host": q.get("classical_host"),
                "classical_mate": q.get("classical_mate"),
                "classical_shared": q.get("classical_shared"),
                "quantum_host": q.get("quantum_host"),
                "quantum_mate": q.get("quantum_mate"),
                "quantum_shared": q.get("quantum_shared"),
            })
    return pd.DataFrame(rows)

telemetry = pd.concat([load_outputs(p) for p in json_paths], ignore_index=True)
telemetry["timestamp"] = pd.to_datetime(telemetry["timestamp"], errors="coerce")
telemetry["cycle"] = pd.to_numeric(telemetry["cycle"], errors="coerce")
telemetry = telemetry.dropna(subset=["timestamp","script_name","cycle",
                                     "classical_host","classical_mate","classical_shared",
                                     "quantum_host","quantum_mate","quantum_shared"]).copy()
telemetry = telemetry.sort_values(["timestamp","script_name","cycle"]).reset_index(drop=True)

print("rows:", len(telemetry), "scripts:", telemetry["script_name"].nunique())
telemetry.head()


rows: 49 scripts: 5


,source_file,script_name,timestamp,cycle,classical_host,classical_mate,classical_shared,quantum_host,quantum_mate,quantum_shared
0,all_data_20250817_095650.json,model,2025-08-17 09:56:43.222236,1,0.714,0.733,0.709,0.376,0.328,0.240
1,all_data_20250817_095650.json,cookie,2025-08-17 09:56:43.749270,1,0.562,0.710,0.603,0.272,0.288,0.216
2,all_data_20250817_095650.json,client,2025-08-17 09:56:44.239768,1,0.468,0.714,0.349,0.288,0.312,0.224
3,all_data_20250817_095650.json,model,2025-08-17 09:56:45.642759,1,0.714,0.733,0.709,0.376,0.328,0.240
4,all_data_20250817_095650.json,model,2025-08-17 09:56:45.643461,2,0.714,0.733,0.709,0.376,0.328,0.240


In [31]:
import json, glob, os
import pandas as pd
import numpy as np

json_paths = sorted(glob.glob("/content/all_data_20250817_*.json"))
assert json_paths, "Upload all_data_20250817_*.json files to /content."

def load_outputs(path):
    with open(path, "r") as f:
        data = json.load(f)
    outputs = data.get("outputs", [])
    rows = []
    for o in outputs:
        pdct = o.get("parsed_data") or {}
        q = pdct.get("quantum_sentiment")
        if q:
            rows.append({
                "source_file": os.path.basename(path),
                "script_name": o.get("script_name"),
                "timestamp": q.get("timestamp") or o.get("timestamp"),
                "cycle": q.get("cycle") or o.get("cycle"),
                "classical_host": q.get("classical_host"),
                "classical_mate": q.get("classical_mate"),
                "classical_shared": q.get("classical_shared"),
                "quantum_host": q.get("quantum_host"),
                "quantum_mate": q.get("quantum_mate"),
                "quantum_shared": q.get("quantum_shared"),
            })
    return pd.DataFrame(rows)

telemetry = pd.concat([load_outputs(p) for p in json_paths], ignore_index=True)
telemetry["timestamp"] = pd.to_datetime(telemetry["timestamp"], errors="coerce")
telemetry["cycle"] = pd.to_numeric(telemetry["cycle"], errors="coerce")
telemetry = telemetry.dropna(subset=["timestamp","script_name","cycle",
                                     "classical_host","classical_mate","classical_shared",
                                     "quantum_host","quantum_mate","quantum_shared"]).copy()
telemetry = telemetry.sort_values(["timestamp","script_name","cycle"]).reset_index(drop=True)

print("rows:", len(telemetry), "scripts:", telemetry["script_name"].nunique())
telemetry.head()


rows: 49 scripts: 5


,source_file,script_name,timestamp,cycle,classical_host,classical_mate,classical_shared,quantum_host,quantum_mate,quantum_shared
0,all_data_20250817_095650.json,model,2025-08-17 09:56:43.222236,1,0.714,0.733,0.709,0.376,0.328,0.240
1,all_data_20250817_095650.json,cookie,2025-08-17 09:56:43.749270,1,0.562,0.710,0.603,0.272,0.288,0.216
2,all_data_20250817_095650.json,client,2025-08-17 09:56:44.239768,1,0.468,0.714,0.349,0.288,0.312,0.224
3,all_data_20250817_095650.json,model,2025-08-17 09:56:45.642759,1,0.714,0.733,0.709,0.376,0.328,0.240
4,all_data_20250817_095650.json,model,2025-08-17 09:56:45.643461,2,0.714,0.733,0.709,0.376,0.328,0.240


In [32]:
import cirq

# Try qsimcirq (fast). Fallback to cirq.Simulator if not available.
try:
    import qsimcirq
    HAVE_QSIM = True
except Exception:
    HAVE_QSIM = False

def make_circuit_from_row(row, n_qubits=6):
    v = np.array([row["quantum_host"], row["quantum_mate"], row["quantum_shared"]], dtype=float)
    v = np.clip(v, -1.0, 1.0)
    angles = (v + 1.0) * np.pi  # [0, 2π]

    qs = cirq.LineQubit.range(n_qubits)
    c = cirq.Circuit()

    for i, qb in enumerate(qs):
        a = angles[i % 3]
        c.append(cirq.ry(a)(qb))
        c.append(cirq.rz(a/2)(qb))
    for i in range(n_qubits-1):
        c.append(cirq.CNOT(qs[i], qs[i+1]))
    c.append(cirq.measure(*qs, key="m"))
    return c, qs

def sample_bits(circuit, reps=256):
    if HAVE_QSIM:
        sim = qsimcirq.QSimSimulator()
    else:
        sim = cirq.Simulator()
    res = sim.run(circuit, repetitions=reps)
    bits = res.measurements["m"].astype(np.uint8)  # (reps, n_qubits)
    return bits

def bits_to_hist(bits):
    # convert each measurement to integer 0..(2^n-1), then histogram
    reps, n = bits.shape
    vals = (bits * (2 ** np.arange(n)[None, :])).sum(axis=1)
    hist = np.bincount(vals, minlength=2**n).astype(np.float32)
    hist /= max(1, hist.sum())
    return hist

# quick test
r0 = telemetry.iloc[0].to_dict()
c0, _ = make_circuit_from_row(r0, n_qubits=6)
b0 = sample_bits(c0, reps=256)
f0 = bits_to_hist(b0)
print("feature dim:", f0.shape, "qsim:", HAVE_QSIM)


feature dim: (64,) qsim: True


In [33]:
from tqdm import tqdm

N_QUBITS = 6
REPS = 256
FEAT_DIM = 2**N_QUBITS

features = np.zeros((len(telemetry), FEAT_DIM), dtype=np.float32)

for i in tqdm(range(len(telemetry))):
    row = telemetry.iloc[i].to_dict()
    c, _ = make_circuit_from_row(row, n_qubits=N_QUBITS)
    bits = sample_bits(c, reps=REPS)
    features[i] = bits_to_hist(bits)

print("features:", features.shape)


100%|██████████| 49/49 [00:00<00:00, 277.18it/s]

features: (49, 64)


In [34]:
from collections import defaultdict

telemetry = telemetry.reset_index(drop=True)
telemetry["script_id"] = telemetry["script_name"].astype("category").cat.codes
n_scripts = telemetry["script_id"].nunique()

# group indices by script
by_script = defaultdict(list)
for idx, sid in enumerate(telemetry["script_id"].values):
    by_script[int(sid)].append(idx)

SEQ_LEN = 5  # Reduced window length to allow sequence creation

X_seq = []
Y_next = []
S_seq = []

targets = telemetry[["quantum_host","quantum_mate","quantum_shared"]].values.astype(np.float32)

for sid, idxs in by_script.items():
    # ensure time order already sorted; idxs are in sorted order due to global sort by timestamp+script
    for j in range(0, len(idxs) - SEQ_LEN - 1):
        win = idxs[j:j+SEQ_LEN]
        nxt = idxs[j+SEQ_LEN]
        X_seq.append(features[win])             # (SEQ_LEN, FEAT_DIM)
        Y_next.append(targets[nxt])             # (3,)
        S_seq.append(sid)                       # script id (observer identity)

X_seq = np.stack(X_seq).astype(np.float32)
Y_next = np.stack(Y_next).astype(np.float32)
S_seq = np.array(S_seq, dtype=np.int64)

print("dataset:", X_seq.shape, Y_next.shape, "scripts:", n_scripts)

dataset: (28, 5, 64) (28, 3) scripts: 5


In [35]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

class HiveDataset(Dataset):
    def __init__(self, X, Y, S):
        self.X = torch.from_numpy(X)   # (N, T, D)
        self.Y = torch.from_numpy(Y)   # (N, 3)
        self.S = torch.from_numpy(S)   # (N,)
    def __len__(self): return self.X.shape[0]
    def __getitem__(self, i):
        return self.X[i], self.Y[i], self.S[i]

class TransformerPredictor(nn.Module):
    def __init__(self, feat_dim, n_scripts, d_model=256, nhead=8, num_layers=4, dropout=0.1):
        super().__init__()
        self.script_emb = nn.Embedding(n_scripts, d_model)
        self.in_proj = nn.Linear(feat_dim, d_model)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=4*d_model, dropout=dropout,
            batch_first=True, activation="gelu"
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.out = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 128),
            nn.GELU(),
            nn.Linear(128, 3)
        )

    def forward(self, x, sid):
        # x: (B,T,D)
        h = self.in_proj(x)
        h = h + self.script_emb(sid).unsqueeze(1)  # add observer embedding
        h = self.encoder(h)
        last = h[:, -1, :]
        return self.out(last)

torch_device = "cuda" if torch.cuda.is_available() else "cpu"
model = TransformerPredictor(FEAT_DIM, n_scripts).to(torch_device)
model

TransformerPredictor(
  (script_emb): Embedding(5, 256)
  (in_proj): Linear(in_features=64, out_features=256, bias=True)
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-3): 4 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
        )
        (linear1): Linear(in_features=256, out_features=1024, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=1024, out_features=256, bias=True)
        (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (out): Sequential(
    (0): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    (1): Linear(in_features=256, out_features=128, bias=True)
    (2): GELU(approximate=

In [36]:
from sklearn.model_selection import train_test_split

idx = np.arange(len(X_seq))
train_idx, val_idx = train_test_split(idx, test_size=0.2, random_state=42)

ds_train = HiveDataset(X_seq[train_idx], Y_next[train_idx], S_seq[train_idx])
ds_val   = HiveDataset(X_seq[val_idx],   Y_next[val_idx],   S_seq[val_idx])

dl_train = DataLoader(ds_train, batch_size=128, shuffle=True, num_workers=2, pin_memory=True)
dl_val   = DataLoader(ds_val, batch_size=256, shuffle=False, num_workers=2, pin_memory=True)

opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-2)
loss_fn = nn.SmoothL1Loss()

def eval_loss():
    model.eval()
    tot, n = 0.0, 0
    with torch.no_grad():
        for x,y,sid in dl_val:
            x,y,sid = x.to(torch_device), y.to(torch_device), sid.to(torch_device)
            pred = model(x, sid)
            loss = loss_fn(pred, y)
            tot += float(loss) * x.size(0)
            n += x.size(0)
    return tot/n

for epoch in range(1, 1000):
    model.train()
    tot, n = 0.0, 0
    for x,y,sid in dl_train:
        x,y,sid = x.to(torch_device), y.to(torch_device), sid.to(torch_device)
        opt.zero_grad(set_to_none=True)
        pred = model(x, sid)
        loss = loss_fn(pred, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        tot += float(loss) * x.size(0)
        n += x.size(0)

    print(f"epoch {epoch:02d} train {tot/n:.5f}  val {eval_loss():.5f}")

/tmp/ipython-input-3226539440.py:38: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  tot += float(loss) * x.size(0)


epoch 01 train 0.07576  val 0.03305
epoch 02 train 0.04111  val 0.02261
epoch 03 train 0.01808  val 0.01025
epoch 04 train 0.01025  val 0.00693
epoch 05 train 0.00822  val 0.00978
epoch 06 train 0.00788  val 0.01037
epoch 07 train 0.00843  val 0.00739
epoch 08 train 0.00634  val 0.00464
epoch 09 train 0.00525  val 0.00327
epoch 10 train 0.00424  val 0.00285
epoch 11 train 0.00393  val 0.00298
epoch 12 train 0.00386  val 0.00320
epoch 13 train 0.00346  val 0.00325
epoch 14 train 0.00325  val 0.00334
epoch 15 train 0.00357  val 0.00335
epoch 16 train 0.00323  val 0.00298
epoch 17 train 0.00255  val 0.00247
epoch 18 train 0.00184  val 0.00225
epoch 19 train 0.00198  val 0.00217
epoch 20 train 0.00239  val 0.00209
epoch 21 train 0.00239  val 0.00199
epoch 22 train 0.00226  val 0.00182
epoch 23 train 0.00213  val 0.00163
epoch 24 train 0.00169  val 0.00157
epoch 25 train 0.00181  val 0.00152
epoch 26 train 0.00148  val 0.00139
epoch 27 train 0.00121  val 0.00131
epoch 28 train 0.00121  val 

In [37]:
from brian2 import *

def brian2_decode(pred_vec, duration_ms=200):
    # pred_vec: (3,) floats in [-1,1] roughly (your targets are around 0..1, but we clip)
    v = np.array(pred_vec, dtype=float)
    v = np.clip(v, -1.0, 1.0)

    start_scope()
    defaultclock.dt = 0.1*ms

    N = 64
    tau = 10*ms
    eqs = '''
    dv/dt = (-v + I)/tau : 1
    I : 1
    '''
    G = NeuronGroup(N, eqs, threshold='v>1', reset='v=0', method='euler')

    # map vector to current profile
    # (simple: base + weighted components)
    base = 0.6
    weights = np.linspace(0.5, 1.5, N)
    drive = base + weights*(0.2*v[0] + 0.2*v[1] + 0.2*v[2])
    G.I = drive

    M = SpikeMonitor(G)
    run(duration_ms*ms)

    # return spike count and rate estimate
    count = M.count[:]  # spikes per neuron
    rate_hz = count.mean() / (duration_ms/1000.0)
    return float(rate_hz), count

# demo: run decoder on a sample
model.eval()
x,y,sid = ds_val[0]
with torch.no_grad():
    pred = model(x.unsqueeze(0).to(torch_device), torch.tensor([int(sid)]).to(torch_device)).cpu().numpy()[0]

rate_hz, counts = brian2_decode(pred, duration_ms=200)
rate_hz, pred

WARNING    'v' is an internal variable of group 'neurongroup', but also exists in the run namespace with the value array([0.32656321, 0.35638663, 0.26467553]). The internal variable will be used. [brian2.groups.group.Group.resolve.resolution_conflict]


(0.0, array([0.3265632 , 0.35638663, 0.26467553], dtype=float32))

In [38]:
telemetry.groupby("script_name")["cycle"].count().sort_values(ascending=False)


,cycle
script_name,
cookie,18
model,17
client,11
blockheart,2
brian,1


In [39]:
!pip -q install torch torchvision torchaudio transformers sentence-transformers accelerate \
  opencv-python pillow librosa ffmpeg-python pandas numpy scikit-learn

# optional (if you want CLIP):
!pip -q install ftfy regex tqdm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.2 MB/s eta 0:00:00


In [40]:
{"timestamp":"2025-08-17T09:56:43.222236","script":"model","cycle":1,
 "text":"operator note ...", "image_path":".../frame_0001.jpg",
 "audio_path":".../clip.wav", "video_path":".../clip.mp4"}


{'timestamp': '2025-08-17T09:56:43.222236',
 'script': 'model',
 'cycle': 1,
 'text': 'operator note ...',
 'image_path': '.../frame_0001.jpg',
 'audio_path': '.../clip.wav',
 'video_path': '.../clip.mp4'}

In [41]:
from sentence_transformers import SentenceTransformer
text_teacher = SentenceTransformer("all-MiniLM-L6-v2")  # small + good
text_teacher.eval()


WARNING    /usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
 [py.warnings]
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(



modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)

In [42]:
import torch
from transformers import CLIPProcessor, CLIPModel
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_proc  = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model.eval()


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

CLIPModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (layer_norm2): LayerNorm((512,), eps=1e-05,

In [43]:
from transformers import WhisperProcessor, WhisperModel
whisper_model = WhisperModel.from_pretrained("openai/whisper-small")  # encoder+decoder, we use encoder
whisper_proc  = WhisperProcessor.from_pretrained("openai/whisper-small")
whisper_model.eval()


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

preprocessor_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

normalizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

WhisperModel(
  (encoder): WhisperEncoder(
    (conv1): Conv1d(80, 768, kernel_size=(3,), stride=(1,), padding=(1,))
    (conv2): Conv1d(768, 768, kernel_size=(3,), stride=(2,), padding=(1,))
    (embed_positions): Embedding(1500, 768)
    (layers): ModuleList(
      (0-11): 12 x WhisperEncoderLayer(
        (self_attn): WhisperAttention(
          (k_proj): Linear(in_features=768, out_features=768, bias=False)
          (v_proj): Linear(in_features=768, out_features=768, bias=True)
          (q_proj): Linear(in_features=768, out_features=768, bias=True)
          (out_proj): Linear(in_features=768, out_features=768, bias=True)
        )
        (self_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (activation_fn): GELUActivation()
        (fc1): Linear(in_features=768, out_features=3072, bias=True)
        (fc2): Linear(in_features=3072, out_features=768, bias=True)
        (final_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      

In [44]:
from transformers import WhisperProcessor, WhisperModel
whisper_model = WhisperModel.from_pretrained("openai/whisper-small")  # encoder+decoder, we use encoder
whisper_proc  = WhisperProcessor.from_pretrained("openai/whisper-small")
whisper_model.eval()


WhisperModel(
  (encoder): WhisperEncoder(
    (conv1): Conv1d(80, 768, kernel_size=(3,), stride=(1,), padding=(1,))
    (conv2): Conv1d(768, 768, kernel_size=(3,), stride=(2,), padding=(1,))
    (embed_positions): Embedding(1500, 768)
    (layers): ModuleList(
      (0-11): 12 x WhisperEncoderLayer(
        (self_attn): WhisperAttention(
          (k_proj): Linear(in_features=768, out_features=768, bias=False)
          (v_proj): Linear(in_features=768, out_features=768, bias=True)
          (q_proj): Linear(in_features=768, out_features=768, bias=True)
          (out_proj): Linear(in_features=768, out_features=768, bias=True)
        )
        (self_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (activation_fn): GELUActivation()
        (fc1): Linear(in_features=768, out_features=3072, bias=True)
        (fc2): Linear(in_features=3072, out_features=768, bias=True)
        (final_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      

In [45]:
import numpy as np
from PIL import Image
import librosa
import cv2

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
clip_model = clip_model.to(DEVICE)
whisper_model = whisper_model.to(DEVICE)

@torch.no_grad()
def embed_text(t: str):
    if not t:
        return None
    v = text_teacher.encode([t], normalize_embeddings=True)[0].astype(np.float32)
    return v  # (d,)

@torch.no_grad()
def embed_image(path: str):
    if not path:
        return None
    img = Image.open(path).convert("RGB")
    inputs = clip_proc(images=img, return_tensors="pt").to(DEVICE)
    feats = clip_model.get_image_features(**inputs)
    feats = torch.nn.functional.normalize(feats, dim=-1)
    return feats[0].cpu().numpy().astype(np.float32)  # (512,)

@torch.no_grad()
def embed_audio(path: str, sr=16000, max_sec=20):
    if not path:
        return None
    wav, _ = librosa.load(path, sr=sr, mono=True)
    wav = wav[: sr*max_sec]
    inputs = whisper_proc(wav, sampling_rate=sr, return_tensors="pt")
    input_features = inputs.input_features.to(DEVICE)  # (1, 80, frames)
    enc = whisper_model.encoder(input_features).last_hidden_state  # (1, T, d)
    # pool
    v = enc.mean(dim=1)[0]
    v = torch.nn.functional.normalize(v, dim=-1)
    return v.cpu().numpy().astype(np.float32)  # (d,)

@torch.no_grad()
def embed_video(path: str, n_frames=8):
    if not path:
        return None
    cap = cv2.VideoCapture(path)
    if not cap.isOpened():
        return None
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    idxs = np.linspace(0, max(0, frame_count-1), n_frames).astype(int)

    embs = []
    cur = 0
    want = set(idxs.tolist())
    i = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if i in want:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            img = Image.fromarray(frame)
            inputs = clip_proc(images=img, return_tensors="pt").to(DEVICE)
            feats = clip_model.get_image_features(**inputs)
            feats = torch.nn.functional.normalize(feats, dim=-1)
            embs.append(feats[0].cpu().numpy())
        i += 1
    cap.release()
    if not embs:
        return None
    v = np.mean(np.stack(embs).astype(np.float32), axis=0)
    v = v / (np.linalg.norm(v) + 1e-9)
    return v.astype(np.float32)  # (512,)


In [46]:
import torch
import torch.nn as nn

class MultiModalStudent(nn.Module):
    def __init__(self, d_model=256, nhead=8, num_layers=4,
                 d_text=384, d_img=512, d_vid=512, d_aud=768, d_telem=64,
                 n_scripts=16):
        super().__init__()
        self.script_emb = nn.Embedding(n_scripts, d_model)

        self.p_text  = nn.Linear(d_text, d_model)
        self.p_img   = nn.Linear(d_img, d_model)
        self.p_vid   = nn.Linear(d_vid, d_model)
        self.p_aud   = nn.Linear(d_aud, d_model)
        self.p_tel   = nn.Linear(d_telem, d_model)

        # token type embeddings: 0=TEL,1=TXT,2=IMG,3=AUD,4=VID
        self.type_emb = nn.Embedding(5, d_model)

        enc = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=4*d_model,
            dropout=0.1, batch_first=True, activation="gelu"
        )
        self.tr = nn.TransformerEncoder(enc, num_layers=num_layers)

        # heads
        self.next_q = nn.Linear(d_model, 3)  # predict next (quantum_host,mate,shared)

        # distill heads (optional): match teacher embeddings (project back)
        self.dist_text = nn.Linear(d_model, d_text)
        self.dist_img  = nn.Linear(d_model, d_img)
        self.dist_aud  = nn.Linear(d_model, d_aud)
        self.dist_vid  = nn.Linear(d_model, d_vid)

    def forward(self, tel_seq, txt_seq, img_seq, aud_seq, vid_seq, mask_seq, script_id):
        """
        Inputs are sequences over time T:
          tel_seq: (B,T,d_telem)
          txt_seq/img_seq/aud_seq/vid_seq: (B,T,d_mod) or zeros if missing
          mask_seq: (B,T,5) 1 if present else 0
        We expand each time step into up to 5 tokens and flatten to (B, T*5, d_model).
        """
        B,T,_ = tel_seq.shape

        # project each modality
        tel = self.p_tel(tel_seq) + self.type_emb(torch.zeros((B,T),dtype=torch.long,device=tel_seq.device))
        txt = self.p_text(txt_seq) + self.type_emb(torch.ones((B,T),dtype=torch.long,device=tel_seq.device))
        img = self.p_img(img_seq)  + self.type_emb(torch.full((B,T),2,dtype=torch.long,device=tel_seq.device))
        aud = self.p_aud(aud_seq)  + self.type_emb(torch.full((B,T),3,dtype=torch.long,device=tel_seq.device))
        vid = self.p_vid(vid_seq)  + self.type_emb(torch.full((B,T),4,dtype=torch.long,device=tel_seq.device))

        # add observer/script embedding to all tokens
        s = self.script_emb(script_id).unsqueeze(1).unsqueeze(1)  # (B,1,1,d)
        tel,txt,img,aud,vid = tel+s,txt+s,img+s,aud+s,vid+s

        # stack tokens per time: (B,T,5,d) -> (B,T*5,d)
        tokens = torch.stack([tel,txt,img,aud,vid], dim=2)
        tokens = tokens.reshape(B, T*5, -1)

        # attention mask: True means "ignore"
        present = mask_seq.reshape(B, T*5)  # 1 present, 0 absent
        src_key_padding_mask = (present == 0)

        h = self.tr(tokens, src_key_padding_mask=src_key_padding_mask)

        # use the last TELEMETRY token at final timestep as summary (index = (T-1)*5 + 0)
        idx = (T-1)*5 + 0
        summary = h[:, idx, :]

        next_q = self.next_q(summary)

        # distill: predict teacher embeddings for *this timestep summary* (you can also do per-modality token)
        return next_q, {
            "text": self.dist_text(summary),
            "img":  self.dist_img(summary),
            "aud":  self.dist_aud(summary),
            "vid":  self.dist_vid(summary),
        }


In [47]:
def cosine_loss(a, b, eps=1e-8):
    a = a / (a.norm(dim=-1, keepdim=True) + eps)
    b = b / (b.norm(dim=-1, keepdim=True) + eps)
    return 1.0 - (a*b).sum(dim=-1).mean()


In [48]:
!pip -q install pandas numpy scikit-learn torch torchvision torchaudio transformers sentence-transformers \
  opencv-python pillow librosa ffmpeg-python matplotlib tqdm

# Cirq + qsimcirq + Brian2
!pip -q install cirq qsimcirq brian2


In [49]:
import json, glob, os
import pandas as pd
import numpy as np

json_paths = sorted(glob.glob("/content/all_data_20250817_*.json"))
assert json_paths, "Upload all_data_20250817_*.json files to /content."

def load_outputs(path):
    with open(path, "r") as f:
        data = json.load(f)
    outputs = data.get("outputs", [])
    rows = []
    for o in outputs:
        pdct = o.get("parsed_data") or {}
        q = pdct.get("quantum_sentiment")
        if q:
            rows.append({
                "source_file": os.path.basename(path),
                "script_name": o.get("script_name"),
                "timestamp": q.get("timestamp") or o.get("timestamp"),
                "cycle": q.get("cycle") or o.get("cycle"),
                "classical_host": q.get("classical_host"),
                "classical_mate": q.get("classical_mate"),
                "classical_shared": q.get("classical_shared"),
                "quantum_host": q.get("quantum_host"),
                "quantum_mate": q.get("quantum_mate"),
                "quantum_shared": q.get("quantum_shared"),
            })
    return pd.DataFrame(rows)

telemetry = pd.concat([load_outputs(p) for p in json_paths], ignore_index=True)
telemetry["timestamp"] = pd.to_datetime(telemetry["timestamp"], errors="coerce")
telemetry["cycle"] = pd.to_numeric(telemetry["cycle"], errors="coerce")
telemetry = telemetry.dropna(subset=[
    "timestamp","script_name","cycle",
    "classical_host","classical_mate","classical_shared",
    "quantum_host","quantum_mate","quantum_shared"
]).copy()

telemetry = telemetry.sort_values(["timestamp","script_name","cycle"]).reset_index(drop=True)
telemetry["script_id"] = telemetry["script_name"].astype("category").cat.codes
n_scripts = telemetry["script_id"].nunique()

print("rows:", len(telemetry), "scripts:", n_scripts)
telemetry.head()


rows: 49 scripts: 5


,source_file,script_name,timestamp,cycle,classical_host,classical_mate,classical_shared,quantum_host,quantum_mate,quantum_shared,script_id
0,all_data_20250817_095650.json,model,2025-08-17 09:56:43.222236,1,0.714,0.733,0.709,0.376,0.328,0.240,4
1,all_data_20250817_095650.json,cookie,2025-08-17 09:56:43.749270,1,0.562,0.710,0.603,0.272,0.288,0.216,3
2,all_data_20250817_095650.json,client,2025-08-17 09:56:44.239768,1,0.468,0.714,0.349,0.288,0.312,0.224,2
3,all_data_20250817_095650.json,model,2025-08-17 09:56:45.642759,1,0.714,0.733,0.709,0.376,0.328,0.240,4
4,all_data_20250817_095650.json,model,2025-08-17 09:56:45.643461,2,0.714,0.733,0.709,0.376,0.328,0.240,4


In [50]:
import cirq

try:
    import qsimcirq
    HAVE_QSIM = True
except Exception:
    HAVE_QSIM = False

def make_circuit_from_row(row, n_qubits=6):
    v = np.array([row["quantum_host"], row["quantum_mate"], row["quantum_shared"]], dtype=float)
    v = np.clip(v, -1.0, 1.0)
    angles = (v + 1.0) * np.pi  # [0, 2π]

    qs = cirq.LineQubit.range(n_qubits)
    c = cirq.Circuit()

    for i, qb in enumerate(qs):
        a = angles[i % 3]
        c.append(cirq.ry(a)(qb))
        c.append(cirq.rz(a/2)(qb))
    for i in range(n_qubits-1):
        c.append(cirq.CNOT(qs[i], qs[i+1]))

    c.append(cirq.measure(*qs, key="m"))
    return c

def sample_bits(circuit, reps=256):
    sim = qsimcirq.QSimSimulator() if HAVE_QSIM else cirq.Simulator()
    res = sim.run(circuit, repetitions=reps)
    return res.measurements["m"].astype(np.uint8)

def bits_to_hist(bits):
    reps, n = bits.shape
    vals = (bits * (2 ** np.arange(n)[None, :])).sum(axis=1)
    hist = np.bincount(vals, minlength=2**n).astype(np.float32)
    hist /= max(1, hist.sum())
    return hist

from tqdm import tqdm

N_QUBITS = 6
REPS = 256
FEAT_DIM = 2**N_QUBITS

telemetry_feats = np.zeros((len(telemetry), FEAT_DIM), dtype=np.float32)
for i in tqdm(range(len(telemetry))):
    row = telemetry.iloc[i].to_dict()
    c = make_circuit_from_row(row, n_qubits=N_QUBITS)
    bits = sample_bits(c, reps=REPS)
    telemetry_feats[i] = bits_to_hist(bits)

print("telemetry_feats:", telemetry_feats.shape, "qsim:", HAVE_QSIM)


100%|██████████| 49/49 [00:00<00:00, 333.92it/s]

telemetry_feats: (49, 64) qsim: True


In [51]:
import os, json, math
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont
import wave
import cv2

outdir = Path("/content/synth")
(outdir/"images").mkdir(parents=True, exist_ok=True)
(outdir/"audio").mkdir(parents=True, exist_ok=True)
(outdir/"video").mkdir(parents=True, exist_ok=True)
(outdir/"text").mkdir(parents=True, exist_ok=True)

def synth_text(row):
    # short “operator note” that encodes state
    ch, cm, cs = row["classical_host"], row["classical_mate"], row["classical_shared"]
    qh, qm, qs = row["quantum_host"], row["quantum_mate"], row["quantum_shared"]
    script = row["script_name"]
    cyc = int(row["cycle"])
    # simple narrative
    mood = "stable" if abs((qs - cs)) < 0.1 else ("drifting" if (qs < cs) else "amplifying")
    return (
        f"[{script}] cycle={cyc} mood={mood}. "
        f"classical(H={ch:.3f}, M={cm:.3f}, S={cs:.3f}) "
        f"quantum(H={qh:.3f}, M={qm:.3f}, S={qs:.3f})."
    )

def save_plot_image(row, path_png):
    ch, cm, cs = row["classical_host"], row["classical_mate"], row["classical_shared"]
    qh, qm, qs = row["quantum_host"], row["quantum_mate"], row["quantum_shared"]
    fig = plt.figure(figsize=(4, 3), dpi=120)
    ax = fig.add_subplot(111)
    ax.bar(["c_host","c_mate","c_shared","q_host","q_mate","q_shared"], [ch,cm,cs,qh,qm,qs])
    ax.set_ylim(0, 1)
    ax.set_title(f'{row["script_name"]} c{int(row["cycle"])}')
    fig.tight_layout()
    fig.savefig(path_png)
    plt.close(fig)

def save_audio_tone(row, path_wav, sr=16000, dur=1.0):
    # map Host/Mate/Shared to 3 tone freqs
    qh, qm, qs = float(row["quantum_host"]), float(row["quantum_mate"]), float(row["quantum_shared"])
    base = 220.0
    f1 = base * (1.0 + qh)
    f2 = base * (1.0 + qm) * 1.25
    f3 = base * (1.0 + qs) * 1.5
    t = np.linspace(0, dur, int(sr*dur), endpoint=False)
    sig = (0.33*np.sin(2*np.pi*f1*t) + 0.33*np.sin(2*np.pi*f2*t) + 0.33*np.sin(2*np.pi*f3*t))
    sig = sig / (np.max(np.abs(sig)) + 1e-9)
    pcm = (sig * 32767).astype(np.int16)

    with wave.open(str(path_wav), "wb") as wf:
        wf.setnchannels(1)
        wf.setsampwidth(2)
        wf.setframerate(sr)
        wf.writeframes(pcm.tobytes())

def save_video_from_frames(row, path_mp4, n_frames=12, fps=12):
    # animate by slowly interpolating classical->quantum bars
    ch, cm, cs = float(row["classical_host"]), float(row["classical_mate"]), float(row["classical_shared"])
    qh, qm, qs = float(row["quantum_host"]), float(row["quantum_mate"]), float(row["quantum_shared"])

    W, H = 480, 360
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    vw = cv2.VideoWriter(str(path_mp4), fourcc, fps, (W, H))

    for k in range(n_frames):
        a = k/(n_frames-1)
        h = (1-a)*ch + a*qh
        m = (1-a)*cm + a*qm
        s = (1-a)*cs + a*qs

        # render simple bars with PIL
        img = Image.new("RGB", (W, H), (20, 20, 24))
        draw = ImageDraw.Draw(img)
        draw.text((12, 10), f'{row["script_name"]} c{int(row["cycle"])} a={a:.2f}', fill=(230,230,230))

        vals = [h, m, s]
        labels = ["host", "mate", "shared"]
        x0 = 60
        for i,(lab,val) in enumerate(zip(labels, vals)):
            x = x0 + i*120
            y_base = 320
            bar_h = int(240 * max(0.0, min(1.0, val)))
            draw.rectangle([x, y_base-bar_h, x+60, y_base], fill=(90, 170, 240))
            draw.text((x, y_base+8), lab, fill=(230,230,230))
            draw.text((x, y_base- bar_h - 18), f"{val:.2f}", fill=(230,230,230))

        frame = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)
        vw.write(frame)

    vw.release()

manifest_path = outdir/"manifest.jsonl"
with open(manifest_path, "w") as f:
    for i in tqdm(range(len(telemetry))):
        row = telemetry.iloc[i].to_dict()
        ts = telemetry.iloc[i]["timestamp"].isoformat()
        script = row["script_name"]
        cyc = int(row["cycle"])

        # file naming
        stem = f"{script}_c{cyc:06d}_{i:07d}"
        txt_path = outdir/"text"/f"{stem}.txt"
        img_path = outdir/"images"/f"{stem}.png"
        wav_path = outdir/"audio"/f"{stem}.wav"
        mp4_path = outdir/"video"/f"{stem}.mp4"

        txt = synth_text(row)
        txt_path.write_text(txt, encoding="utf-8")
        save_plot_image(row, img_path)
        save_audio_tone(row, wav_path)
        save_video_from_frames(row, mp4_path)

        rec = {
            "i": i,
            "timestamp": ts,
            "script_name": script,
            "script_id": int(telemetry.iloc[i]["script_id"]),
            "cycle": cyc,
            "text": txt,
            "image_path": str(img_path),
            "audio_path": str(wav_path),
            "video_path": str(mp4_path)
        }
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

100%|██████████| 49/49 [00:17<00:00,  2.75it/s]


In [52]:
import torch
from sentence_transformers import SentenceTransformer
from transformers import CLIPProcessor, CLIPModel, WhisperProcessor, WhisperModel
import librosa
from PIL import Image

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

text_teacher = SentenceTransformer("all-MiniLM-L6-v2")  # 384-d
text_teacher.eval()

clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(DEVICE).eval()
clip_proc  = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

whisper_model = WhisperModel.from_pretrained("openai/whisper-small").to(DEVICE).eval()
whisper_proc  = WhisperProcessor.from_pretrained("openai/whisper-small")


In [53]:
import numpy as np
import cv2
from tqdm import tqdm

def embed_text(t: str):
    v = text_teacher.encode([t], normalize_embeddings=True)[0].astype(np.float32)
    return v  # (384,)

@torch.no_grad()
def embed_image(path: str):
    img = Image.open(path).convert("RGB")
    inputs = clip_proc(images=img, return_tensors="pt").to(DEVICE)
    feats = clip_model.get_image_features(**inputs)
    feats = torch.nn.functional.normalize(feats, dim=-1)
    return feats[0].cpu().numpy().astype(np.float32)  # (512,)

@torch.no_grad()
def embed_audio(path: str, sr=16000, max_sec=2):
    wav, _ = librosa.load(path, sr=sr, mono=True)
    wav = wav[: sr*max_sec]
    inputs = whisper_proc(wav, sampling_rate=sr, return_tensors="pt")
    input_features = inputs.input_features.to(DEVICE)
    enc = whisper_model.encoder(input_features).last_hidden_state  # (1,T,768)
    v = enc.mean(dim=1)[0]
    v = torch.nn.functional.normalize(v, dim=-1)
    return v.cpu().numpy().astype(np.float32)  # (768,)

@torch.no_grad()
def embed_video(path: str, n_frames=8):
    cap = cv2.VideoCapture(path)
    if not cap.isOpened():
        return np.zeros((512,), dtype=np.float32)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    idxs = np.linspace(0, max(0, frame_count-1), n_frames).astype(int)
    want = set(idxs.tolist())

    embs = []
    i = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if i in want:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            img = Image.fromarray(frame)
            inputs = clip_proc(images=img, return_tensors="pt").to(DEVICE)
            feats = clip_model.get_image_features(**inputs)
            feats = torch.nn.functional.normalize(feats, dim=-1)
            embs.append(feats[0].cpu().numpy())
        i += 1
    cap.release()
    if not embs:
        return np.zeros((512,), dtype=np.float32)
    v = np.mean(np.stack(embs).astype(np.float32), axis=0)
    v = v / (np.linalg.norm(v) + 1e-9)
    return v.astype(np.float32)

# Read manifest
records = [json.loads(line) for line in open(manifest_path, "r", encoding="utf-8")]

D_TEXT, D_IMG, D_AUD, D_VID = 384, 512, 768, 512
E_text = np.zeros((len(records), D_TEXT), dtype=np.float32)
E_img  = np.zeros((len(records), D_IMG),  dtype=np.float32)
E_aud  = np.zeros((len(records), D_AUD),  dtype=np.float32)
E_vid  = np.zeros((len(records), D_VID),  dtype=np.float32)

for r in tqdm(records):
    i = r["i"]
    E_text[i] = embed_text(r["text"])
    E_img[i]  = embed_image(r["image_path"])
    E_aud[i]  = embed_audio(r["audio_path"])
    E_vid[i]  = embed_video(r["video_path"])

np.save(outdir/"E_text.npy", E_text)
np.save(outdir/"E_img.npy",  E_img)
np.save(outdir/"E_aud.npy",  E_aud)
np.save(outdir/"E_vid.npy",  E_vid)

print("cached embeddings saved in", outdir)


100%|██████████| 49/49 [00:50<00:00,  1.03s/it]

cached embeddings saved in /content/synth


In [54]:
from collections import defaultdict

targets = telemetry[["quantum_host","quantum_mate","quantum_shared"]].values.astype(np.float32)
script_ids = telemetry["script_id"].values.astype(np.int64)

# modality embeddings
E_text = np.load(outdir/"E_text.npy").astype(np.float32)
E_img  = np.load(outdir/"E_img.npy").astype(np.float32)
E_aud  = np.load(outdir/"E_aud.npy").astype(np.float32)
E_vid  = np.load(outdir/"E_vid.npy").astype(np.float32)

# reduce telemetry feature dim -> d_telem using PCA-ish linear projection (fast)
# (You can replace with nn.Linear in the model; we’ll keep it model-side to stay flexible.)
D_TELEM = telemetry_feats.shape[1]

by_script = defaultdict(list)
for idx, sid in enumerate(script_ids):
    by_script[int(sid)].append(idx)

SEQ_LEN = 5 # Changed from 32 to 5 to allow sequence creation

X_tel, X_txt, X_img, X_aud, X_vid, S_id = [], [], [], [], [], []
Y_next = []
T_txt, T_img, T_aud, T_vid = [], [], [], []

for sid, idxs in by_script.items():
    for j in range(0, len(idxs) - SEQ_LEN - 1):
        win = idxs[j:j+SEQ_LEN]
        nxt = idxs[j+SEQ_LEN]

        X_tel.append(telemetry_feats[win])   # (T, D_TELEM)
        X_txt.append(E_text[win])
        X_img.append(E_img[win])
        X_aud.append(E_aud[win])
        X_vid.append(E_vid[win])

        S_id.append(sid)
        Y_next.append(targets[nxt])

        # distill target = teacher embeddings at last timestep of window
        last = win[-1]
        T_txt.append(E_text[last]); T_img.append(E_img[last]); T_aud.append(E_aud[last]); T_vid.append(E_vid[last])

# Only stack if lists are not empty
if X_tel:
    X_tel = np.stack(X_tel).astype(np.float32)
    X_txt = np.stack(X_txt).astype(np.float32)
    X_img = np.stack(X_img).astype(np.float32)
    X_aud = np.stack(X_aud).astype(np.float32)
    X_vid = np.stack(X_vid).astype(np.float32)
    S_id  = np.array(S_id, dtype=np.int64)

    Y_next = np.stack(Y_next).astype(np.float32)
    T_txt  = np.stack(T_txt).astype(np.float32)
    T_img  = np.stack(T_img).astype(np.float32)
    T_aud  = np.stack(T_aud).astype(np.float32)
    T_vid  = np.stack(T_vid).astype(np.float32)

    print("seq dataset:", X_tel.shape, Y_next.shape, "scripts:", n_scripts)
else:
    print("No sequences could be created with the current SEQ_LEN and data.")
    X_tel, X_txt, X_img, X_aud, X_vid, S_id, Y_next, T_txt, T_img, T_aud, T_vid = (
        np.array([]) for _ in range(11)
    )

seq dataset: (28, 5, 64) (28, 3) scripts: 5


In [55]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

class MMSeqDataset(Dataset):
    def __init__(self, X_tel, X_txt, X_img, X_aud, X_vid, S_id, Y_next, T_txt, T_img, T_aud, T_vid):
        self.X_tel = torch.from_numpy(X_tel)
        self.X_txt = torch.from_numpy(X_txt)
        self.X_img = torch.from_numpy(X_img)
        self.X_aud = torch.from_numpy(X_aud)
        self.X_vid = torch.from_numpy(X_vid)
        self.S_id  = torch.from_numpy(S_id)
        self.Y_next= torch.from_numpy(Y_next)
        self.T_txt = torch.from_numpy(T_txt)
        self.T_img = torch.from_numpy(T_img)
        self.T_aud = torch.from_numpy(T_aud)
        self.T_vid = torch.from_numpy(T_vid)

    def __len__(self): return self.X_tel.shape[0]
    def __getitem__(self, i):
        return (self.X_tel[i], self.X_txt[i], self.X_img[i], self.X_aud[i], self.X_vid[i],
                self.S_id[i], self.Y_next[i], self.T_txt[i], self.T_img[i], self.T_aud[i], self.T_vid[i])

def cosine_loss(a, b, eps=1e-8):
    a = a / (a.norm(dim=-1, keepdim=True) + eps)
    b = b / (b.norm(dim=-1, keepdim=True) + eps)
    return 1.0 - (a*b).sum(dim=-1).mean()

class MultiModalStudent(nn.Module):
    def __init__(self, d_telem, n_scripts, d_model=256, nhead=8, num_layers=4,
                 d_text=384, d_img=512, d_vid=512, d_aud=768, dropout=0.1):
        super().__init__()
        self.script_emb = nn.Embedding(n_scripts, d_model)

        self.p_tel  = nn.Linear(d_telem, d_model)
        self.p_text = nn.Linear(d_text, d_model)
        self.p_img  = nn.Linear(d_img, d_model)
        self.p_aud  = nn.Linear(d_aud, d_model)
        self.p_vid  = nn.Linear(d_vid, d_model)

        self.type_emb = nn.Embedding(5, d_model)  # 0=TEL 1=TXT 2=IMG 3=AUD 4=VID

        enc = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=4*d_model,
            dropout=dropout, batch_first=True, activation="gelu"
        )
        self.tr = nn.TransformerEncoder(enc, num_layers=num_layers)

        self.next_q = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, 3))

        self.dist_text = nn.Linear(d_model, d_text)
        self.dist_img  = nn.Linear(d_model, d_img)
        self.dist_aud  = nn.Linear(d_model, d_aud)
        self.dist_vid  = nn.Linear(d_model, d_vid)

    def forward(self, tel_seq, txt_seq, img_seq, aud_seq, vid_seq, script_id):
        B,T,_ = tel_seq.shape
        s = self.script_emb(script_id).unsqueeze(1).unsqueeze(1)  # (B,1,1,d)

        tel = self.p_tel(tel_seq)  + self.type_emb(torch.zeros((B,T),dtype=torch.long,device=tel_seq.device)) + s.squeeze(1)
        txt = self.p_text(txt_seq) + self.type_emb(torch.ones((B,T),dtype=torch.long,device=tel_seq.device)) + s.squeeze(1)
        img = self.p_img(img_seq)  + self.type_emb(torch.full((B,T),2,dtype=torch.long,device=tel_seq.device)) + s.squeeze(1)
        aud = self.p_aud(aud_seq)  + self.type_emb(torch.full((B,T),3,dtype=torch.long,device=tel_seq.device)) + s.squeeze(1)
        vid = self.p_vid(vid_seq)  + self.type_emb(torch.full((B,T),4,dtype=torch.long,device=tel_seq.device)) + s.squeeze(1)

        tokens = torch.stack([tel,txt,img,aud,vid], dim=2).reshape(B, T*5, -1)
        h = self.tr(tokens)

        summary = h[:, (T-1)*5 + 0, :]  # last timestep telemetry token
        next_q = self.next_q(summary)

        dist = {
            "text": self.dist_text(summary),
            "img":  self.dist_img(summary),
            "aud":  self.dist_aud(summary),
            "vid":  self.dist_vid(summary),
        }
        return next_q, dist

torch_device = "cuda" if torch.cuda.is_available() else "cpu"

idx = np.arange(len(X_tel))
tr_idx, va_idx = train_test_split(idx, test_size=0.2, random_state=42)

ds_tr = MMSeqDataset(X_tel[tr_idx], X_txt[tr_idx], X_img[tr_idx], X_aud[tr_idx], X_vid[tr_idx],
                    S_id[tr_idx], Y_next[tr_idx], T_txt[tr_idx], T_img[tr_idx], T_aud[tr_idx], T_vid[tr_idx])
ds_va = MMSeqDataset(X_tel[va_idx], X_txt[va_idx], X_img[va_idx], X_aud[va_idx], X_vid[va_idx],
                    S_id[va_idx], Y_next[va_idx], T_txt[va_idx], T_img[va_idx], T_aud[va_idx], T_vid[va_idx])

dl_tr = DataLoader(ds_tr, batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
dl_va = DataLoader(ds_va, batch_size=128, shuffle=False, num_workers=2, pin_memory=True)

model = MultiModalStudent(d_telem=D_TELEM, n_scripts=n_scripts).to(torch_device)

opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-2)
loss_next = nn.SmoothL1Loss()

w_txt, w_img, w_aud, w_vid = 0.2, 0.2, 0.2, 0.2

def eval_one():
    model.eval()
    tot = 0.0
    n = 0
    with torch.no_grad():
        for batch in dl_va:
            tel, txt, img, aud, vid, sid, y, tt, ti, ta, tv = batch
            tel, txt, img, aud, vid = tel.to(torch_device), txt.to(torch_device), img.to(torch_device), aud.to(torch_device), vid.to(torch_device)
            sid, y = sid.to(torch_device), y.to(torch_device)
            tt, ti, ta, tv = tt.to(torch_device), ti.to(torch_device), ta.to(torch_device), tv.to(torch_device)

            pred_q, dist = model(tel, txt, img, aud, vid, sid)
            L = loss_next(pred_q, y)
            L = L + w_txt*cosine_loss(dist["text"], tt)
            L = L + w_img*cosine_loss(dist["img"],  ti)
            L = L + w_aud*cosine_loss(dist["aud"],  ta)
            L = L + w_vid*cosine_loss(dist["vid"],  tv)

            tot += float(L) * tel.size(0)
            n += tel.size(0)
    return tot/n

for epoch in range(1, 15000):
    model.train()
    tot = 0.0
    n = 0
    for batch in dl_tr:
        tel, txt, img, aud, vid, sid, y, tt, ti, ta, tv = batch
        tel, txt, img, aud, vid = tel.to(torch_device), txt.to(torch_device), img.to(torch_device), aud.to(torch_device), vid.to(torch_device)
        sid, y = sid.to(torch_device), y.to(torch_device)
        tt, ti, ta, tv = tt.to(torch_device), ti.to(torch_device), ta.to(torch_device), tv.to(torch_device)

        opt.zero_grad(set_to_none=True)
        pred_q, dist = model(tel, txt, img, aud, vid, sid)

        L = loss_next(pred_q, y)
        L = L + w_txt*cosine_loss(dist["text"], tt)
        L = L + w_img*cosine_loss(dist["img"],  ti)
        L = L + w_aud*cosine_loss(dist["aud"],  ta)
        L = L + w_vid*cosine_loss(dist["vid"],  tv)

        L.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()

        tot += float(L) * tel.size(0)
        n += tel.size(0)

    print(f"epoch {epoch:02d} train {tot/n:.5f}  val {eval_one():.5f}")

Streaming output truncated to the last 5000 lines.
epoch 10000 train 0.00003  val 0.00596
epoch 10001 train 0.00003  val 0.00592
epoch 10002 train 0.00004  val 0.00551
epoch 10003 train 0.00003  val 0.00532
epoch 10004 train 0.00003  val 0.00515
epoch 10005 train 0.00003  val 0.00524
epoch 10006 train 0.00004  val 0.00555
epoch 10007 train 0.00003  val 0.00544
epoch 10008 train 0.00004  val 0.00508
epoch 10009 train 0.00005  val 0.00535
epoch 10010 train 0.00005  val 0.00546
epoch 10011 train 0.00004  val 0.00546
epoch 10012 train 0.00004  val 0.00583
epoch 10013 train 0.00004  val 0.00602
epoch 10014 train 0.00004  val 0.00578
epoch 10015 train 0.00004  val 0.00561
epoch 10016 train 0.00004  val 0.00573
epoch 10017 train 0.00004  val 0.00562
epoch 10018 train 0.00005  val 0.00506
epoch 10019 train 0.00005  val 0.00487
epoch 10020 train 0.00004  val 0.00502
epoch 10021 train 0.00004  val 0.00506
epoch 10022 train 0.00003  val 0.00504
epoch 10023 train 0.00004  val 0.00556
epoch 10024 t

In [56]:
from brian2 import *

def brian2_decode(pred_vec, duration_ms=200):
    v = np.array(pred_vec, dtype=float)
    v = np.clip(v, -1.0, 1.0)

    start_scope()
    defaultclock.dt = 0.1*ms
    N = 64
    tau = 10*ms
    eqs = '''
    dv/dt = (-v + I)/tau : 1
    I : 1
    '''
    G = NeuronGroup(N, eqs, threshold='v>1', reset='v=0', method='euler')

    base = 0.6
    weights = np.linspace(0.5, 1.5, N)
    drive = base + weights*(0.2*v[0] + 0.2*v[1] + 0.2*v[2])
    G.I = drive

    M = SpikeMonitor(G)
    run(duration_ms*ms)

    rate_hz = M.num_spikes / (N * (duration_ms/1000.0))
    return float(rate_hz), M.count[:]

# demo a single sample from val set
model.eval()
batch = next(iter(dl_va))
tel, txt, img, aud, vid, sid, y, tt, ti, ta, tv = batch
with torch.no_grad():
    pred_q, _ = model(tel[:1].to(torch_device), txt[:1].to(torch_device), img[:1].to(torch_device), aud[:1].to(torch_device), vid[:1].to(torch_device), sid[:1].to(torch_device))
pred = pred_q.cpu().numpy()[0]
rate_hz, counts = brian2_decode(pred, duration_ms=200)
pred, rate_hz

WARNING    'v' is an internal variable of group 'neurongroup', but also exists in the run namespace with the value array([0.26342431, 0.32777533, 0.32145607]). The internal variable will be used. [brian2.groups.group.Group.resolve.resolution_conflict]


(array([0.2634243 , 0.32777533, 0.32145607], dtype=float32), 0.0)

In [57]:
# Colab cell
!pip -q install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip -q install numpy pandas scikit-learn tqdm matplotlib
!pip -q install cirq-core qsimcirq
# brian2 is optional (only if you really need spiking dynamics)
!pip -q install brian2


In [60]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [61]:
!mkdir -p /content/data
!unzip -q *.zip -d /content/data
!ls -lah /content/data


unzip:  cannot find or open *.zip, *.zip.zip or *.zip.ZIP.

No zipfiles found.
total 8.0K
drwxr-xr-x 2 root root 4.0K Jan 19 20:23 .
drwxr-xr-x 1 root root 4.0K Jan 19 20:23 ..


In [62]:
import os, glob
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

class HiveDataset(Dataset):
    def __init__(self, root="/content/data", split="train"):
        # Example: look for npz files like train.npz / val.npz
        npz_path = os.path.join(root, f"{split}.npz")
        if os.path.exists(npz_path):
            d = np.load(npz_path, allow_pickle=True)
            self.X = d["X"].astype(np.float32)
            self.y = d["y"]
            self.obs = d["obs"].astype(np.int64) if "obs" in d else None
            self.seq = (self.X.ndim == 3)  # (N,T,D) vs (N,D)
            return

        # Fallback: you can adapt parsing here
        raise FileNotFoundError(f"Expected {npz_path}. Export your data to train.npz/val.npz with keys X,y,(obs).")

    def __len__(self): return len(self.X)

    def __getitem__(self, i):
        x = torch.from_numpy(self.X[i])
        y = torch.tensor(self.y[i]).long() if np.issubdtype(self.y.dtype, np.integer) else torch.tensor(self.y[i]).float()
        if self.obs is None:
            obs = torch.tensor(0).long()
        else:
            obs = torch.tensor(self.obs[i]).long()
        return x, obs, y


In [63]:
import torch.nn as nn
import torch.nn.functional as F

class MLPEncoder(nn.Module):
    def __init__(self, d_in, z_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, 512), nn.GELU(), nn.Dropout(0.1),
            nn.Linear(512, 256), nn.GELU(), nn.Dropout(0.1),
            nn.Linear(256, z_dim),
            nn.LayerNorm(z_dim)
        )
    def forward(self, x):
        return self.net(x)


In [64]:
class TransformerEncoder(nn.Module):
    def __init__(self, d_in, h=256, z_dim=128, layers=4, heads=4, max_len=512):
        super().__init__()
        self.proj = nn.Linear(d_in, h)
        self.pos = nn.Embedding(max_len, h)
        enc_layer = nn.TransformerEncoderLayer(d_model=h, nhead=heads, batch_first=True, dropout=0.1, norm_first=True)
        self.enc = nn.TransformerEncoder(enc_layer, num_layers=layers)
        self.to_z = nn.Sequential(nn.Linear(h, z_dim), nn.LayerNorm(z_dim))

    def forward(self, x):  # x: (B,T,D)
        B,T,D = x.shape
        t = torch.arange(T, device=x.device).unsqueeze(0).expand(B,T)
        h = self.proj(x) + self.pos(t)
        h = self.enc(h)
        pooled = h.mean(dim=1)
        return self.to_z(pooled)


In [65]:
class FiLM(nn.Module):
    def __init__(self, z_dim, n_obs=3, hidden=128):
        super().__init__()
        self.emb = nn.Embedding(n_obs, hidden)
        self.to_gamma = nn.Linear(hidden, z_dim)
        self.to_beta  = nn.Linear(hidden, z_dim)

    def forward(self, z, obs_id):
        h = self.emb(obs_id)
        gamma = self.to_gamma(h)
        beta = self.to_beta(h)
        return gamma * z + beta


In [66]:
from tqdm import tqdm

def train_one_epoch(model, loader, opt, device, task="classify"):
    model.train()
    total, correct, loss_sum = 0, 0, 0.0
    for x, obs, y in tqdm(loader, leave=False):
        x, obs, y = x.to(device), obs.to(device), y.to(device)
        opt.zero_grad()

        logits, z = model(x, obs)

        if task == "classify":
            loss = F.cross_entropy(logits, y)
            pred = logits.argmax(dim=-1)
            correct += (pred == y).sum().item()
            total += y.numel()
        else:
            # Corrected: Removed y.view(-1,1) as y is already (batch_size, 3) and logits will be (batch_size, 3)
            loss = F.mse_loss(logits, y)
            total += y.numel()

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        loss_sum += loss.item() * x.size(0)

    acc = correct/total if task=="classify" else None
    return loss_sum/len(loader.dataset), acc

@torch.no_grad()
def eval_one_epoch(model, loader, device, task="classify"):
    model.eval()
    total, correct, loss_sum = 0, 0, 0.0
    for x, obs, y in loader:
        x, obs, y = x.to(device), obs.to(device), y.to(device)
        logits, z = model(x, obs)
        if task == "classify":
            loss = F.cross_entropy(logits, y)
            pred = logits.argmax(dim=-1)
            correct += (pred == y).sum().item()
            total += y.numel()
        else:
            # Corrected: Removed y.view(-1,1)
            loss = F.mse_loss(logits, y)
            total += y.numel()
        loss_sum += loss.item() * x.size(0)
    acc = correct/total if task=="classify" else None
    return loss_sum/len(loader.dataset), acc

In [67]:
from tqdm import tqdm

def train_one_epoch(model, loader, opt, device, task="classify"):
    model.train()
    total, correct, loss_sum = 0, 0, 0.0
    for x, obs, y in tqdm(loader, leave=False):
        x, obs, y = x.to(device), obs.to(device), y.to(device)
        opt.zero_grad()

        logits, z = model(x, obs)

        if task == "classify":
            loss = F.cross_entropy(logits, y)
            pred = logits.argmax(dim=-1)
            correct += (pred == y).sum().item()
            total += y.numel()
        else:
            # Corrected: Removed y.view(-1,1) as y is already (batch_size, 3) and logits will be (batch_size, 3)
            loss = F.mse_loss(logits, y)
            total += y.numel()

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        loss_sum += loss.item() * x.size(0)

    acc = correct/total if task=="classify" else None
    return loss_sum/len(loader.dataset), acc

@torch.no_grad()
def eval_one_epoch(model, loader, device, task="classify"):
    model.eval()
    total, correct, loss_sum = 0, 0, 0.0
    for x, obs, y in loader:
        x, obs, y = x.to(device), obs.to(device), y.to(device)
        logits, z = model(x, obs)
        if task == "classify":
            loss = F.cross_entropy(logits, y)
            pred = logits.argmax(dim=-1)
            correct += (pred == y).sum().item()
            total += y.numel()
        else:
            # Corrected: Removed y.view(-1,1)
            loss = F.mse_loss(logits, y)
            total += y.numel()
        loss_sum += loss.item() * x.size(0)
    acc = correct/total if task=="classify" else None
    return loss_sum/len(loader.dataset), acc

In [68]:
device = "cuda" if torch.cuda.is_available() else "cpu"

class HiveModel(nn.Module):
    def __init__(self, encoder, z_dim, n_obs, n_classes, task="classify"):
        super().__init__()
        self.encoder = encoder
        self.film = FiLM(z_dim, n_obs)
        self.task = task

        if task == "classify":
            self.head = nn.Linear(z_dim, n_classes)
        elif task == "regress":
            # Corrected: Output n_classes (3) for regression
            self.head = nn.Linear(z_dim, n_classes)
        else:
            raise ValueError("Task must be 'classify' or 'regress'")

    def forward(self, x, obs_id):
        z = self.encoder(x)
        z_conditioned = self.film(z, obs_id)
        logits = self.head(z_conditioned)
        return logits, z_conditioned

In [69]:
import os
from sklearn.model_selection import train_test_split

# Create /content/data if it doesn't exist
os.makedirs("/content/data", exist_ok=True)

# Assuming X_tel, Y_next, S_id are already defined from previous cells
# Handle the case where X_tel might be empty if no sequences were formed
if X_tel.size == 0:
    print("No data to save. X_tel is empty.")
else:
    # Split data into train and validation sets
    X_train_data, X_val_data, y_train_data, y_val_data, obs_train_data, obs_val_data = \
        train_test_split(X_tel, Y_next, S_id, test_size=0.2, random_state=42, stratify=S_id)

    # Save training data
    np.savez_compressed(
        "/content/data/train.npz",
        X=X_train_data,
        y=y_train_data,
        obs=obs_train_data
    )
    print(f"Saved /content/data/train.npz with X shape {X_train_data.shape}, y shape {y_train_data.shape}, obs shape {obs_train_data.shape}")

    # Save validation data
    np.savez_compressed(
        "/content/data/val.npz",
        X=X_val_data,
        y=y_val_data,
        obs=obs_val_data
    )
    print(f"Saved /content/data/val.npz with X shape {X_val_data.shape}, y shape {y_val_data.shape}, obs shape {obs_val_data.shape}")

# Now, proceed with model training after ensuring data is present

train_ds = HiveDataset("/content/data", "train")
val_ds   = HiveDataset("/content/data", "val")

train_loader = DataLoader(train_ds, batch_size=256, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=256, shuffle=False, num_workers=2, pin_memory=True)

# detect shapes
x0, obs0, y0 = train_ds[0]
seq = (x0.ndim == 2)  # (T,D) => sequence

n_obs = train_ds.obs.max().item() + 1 if train_ds.obs is not None else 1 # dynamically get n_obs
# The target is an array of 3 floats, so it's a regression task
task = "regress" # Changed from classify

if not seq:
    d_in = x0.shape[0]
    encoder = MLPEncoder(d_in=d_in, z_dim=128)
else:
    T, d_in = x0.shape
    encoder = TransformerEncoder(d_in=d_in, h=256, z_dim=128, layers=4, heads=4, max_len=SEQ_LEN)

model = HiveModel(encoder=encoder, z_dim=128, n_obs=n_obs, n_classes=3, task=task).to(device) # n_classes=3 for quantum_host,mate,shared

opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-2)

best = 1e9
for epoch in range(1, 1000):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, opt, device, task=task)
    va_loss, va_acc = eval_one_epoch(model, val_loader, device, task=task)

    print(f"Epoch {epoch:02d} | train loss {tr_loss:.4f} val loss {va_loss:.4f}"
          + (f" | train acc {tr_acc:.3f} val acc {va_acc:.3f}" if task=="classify" else ""))

    if va_loss < best:
        best = va_loss
        torch.save(model.state_dict(), "/content/drive/MyDrive/hive_best.pt" if os.path.exists("/content/drive") else "/content/hive_best.pt")

WARNING    /usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(
 [py.warnings]
  warnings.warn(



Saved /content/data/train.npz with X shape (22, 5, 64), y shape (22, 3), obs shape (22,)
Saved /content/data/val.npz with X shape (6, 5, 64), y shape (6, 3), obs shape (6,)


Epoch 01 | train loss 0.1873 val loss 0.4979


Epoch 02 | train loss 0.4372 val loss 0.1891


Epoch 03 | train loss 0.1612 val loss 0.0633


...

Epoch 995 | train loss 0.0004 val loss 0.0005


Epoch 996 | train loss 0.0004 val loss 0.0011


Epoch 997 | train loss 0.0006 val loss 0.0004


Epoch 998 | train loss 0.0005 val loss 0.0006


Epoch 999 | train loss 0.0004 val loss 0.0008


In [70]:
!pip -q install torch numpy scikit-learn tqdm


In [71]:
import os, json, re
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedShuffleSplit

ROOT = "/content/synth"  # Corrected path

mood_re = re.compile(r"mood=([a-zA-Z_]+)")
cls_re  = re.compile(r"classical\(H=([0-9.]+),\s*M=([0-9.]+),\s*S=([0-9.]+)\)")
q_re    = re.compile(r"quantum\(H=([0-9.]+),\s*M=([0-9.]+),\s*S=([0-9.]+)\)")

def load_manifest(path):
    rows = []
    print(f"Attempting to load manifest from: {path}")
    if not os.path.exists(path):
        print(f"Error: Path does not exist: {path}")
        # Add another check with `!ls -la` directly here to compare within Python context
        !ls -la {os.path.dirname(path)}
        raise FileNotFoundError(f"Manifest file not found at {path}")
    with open(path, "r") as f:
        for line in f:
            d = json.loads(line)
            t = d.get("text","")
            mood = mood_re.search(t).group(1)
            c = cls_re.search(t)
            q = q_re.search(t)
            classical = np.array([float(c.group(1)), float(c.group(2)), float(c.group(3))], dtype=np.float32)
            quantum   = np.array([float(q.group(1)), float(q.group(2)), float(q.group(3))], dtype=np.float32)
            rows.append({
                "i": d["i"],
                "script_name": d["script_name"],
                "cycle": d["cycle"],
                "mood": mood,
                "classical": classical,
                "quantum": quantum
            })
    return rows

rows = load_manifest(os.path.join(ROOT, "manifest.jsonl"))
print("rows:", len(rows), "unique scripts:", sorted(set(r["script_name"] for r in rows)), "unique moods:", sorted(set(r["mood"] for r in rows)))

E_text = np.load(os.path.join(ROOT, "E_text.npy")).astype(np.float32)
E_img  = np.load(os.path.join(ROOT, "E_img.npy")).astype(np.float32)
E_vid  = np.load(os.path.join(ROOT, "E_vid.npy")).astype(np.float32)
E_aud  = np.load(os.path.join(ROOT, "E_aud.npy")).astype(np.float32)

assert E_text.shape[0] == len(rows) == E_img.shape[0] == E_vid.shape[0] == E_aud.shape[0]

# label maps
scripts = sorted(set(r["script_name"] for r in rows))
script2id = {s:i for i,s in enumerate(scripts)}

moods = sorted(set(r["mood"] for r in rows))
mood2id = {m:i for i,m in enumerate(moods)}

y_mood = np.array([mood2id[r["mood"]] for r in rows], dtype=np.int64)
obs_id = np.array([script2id[r["script_name"]] for r in rows], dtype=np.int64)

Y_classical = np.stack([r["classical"] for r in rows], axis=0)
Y_quantum   = np.stack([r["quantum"] for r in rows], axis=0)

# stratified split by mood (since tiny dataset)
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, val_idx = next(sss.split(np.zeros(len(y_mood)), y_mood))
print("train/val:", len(train_idx), len(val_idx))

# Define n_obs and n_moods globally
n_obs = len(scripts)
n_moods = len(moods)

Attempting to load manifest from: /content/synth/manifest.jsonl
rows: 49 unique scripts: ['blockheart', 'brian', 'client', 'cookie', 'model'] unique moods: ['drifting', 'stable']
train/val: 36 13


In [72]:
import os

file_path = "/content/data/synth/manifest.jsonl"
if os.path.exists(file_path):
    print(f"The file {file_path} exists.")
else:
    print(f"The file {file_path} does NOT exist.")

The file /content/data/synth/manifest.jsonl does NOT exist.


In [73]:
class SynthHive(Dataset):
    def __init__(self, idx):
        self.idx = idx

    def __len__(self): return len(self.idx)

    def __getitem__(self, k):
        i = self.idx[k]
        x = {
            "text": torch.from_numpy(E_text[i]),
            "img":  torch.from_numpy(E_img[i]),
            "vid":  torch.from_numpy(E_vid[i]),
            "aud":  torch.from_numpy(E_aud[i]),
        }
        obs = torch.tensor(obs_id[i]).long()
        mood = torch.tensor(y_mood[i]).long()
        classical = torch.from_numpy(Y_classical[i])
        quantum   = torch.from_numpy(Y_quantum[i])
        return x, obs, mood, classical, quantum

train_ds = SynthHive(train_idx)
val_ds   = SynthHive(val_idx)
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=16, shuffle=False)


In [74]:
import torch.nn as nn
import torch.nn.functional as F

class ModEncoder(nn.Module):
    def __init__(self, d_in, z=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, 256), nn.GELU(), nn.Dropout(0.1),
            nn.Linear(256, z),
            nn.LayerNorm(z)
        )
    def forward(self, x): return self.net(x)

class FiLM(nn.Module):
    def __init__(self, z=128, n_obs=5, h=128):
        super().__init__()
        self.emb = nn.Embedding(n_obs, h)
        self.gamma = nn.Linear(h, z)
        self.beta  = nn.Linear(h, z)
    def forward(self, zvec, obs):
        e = self.emb(obs)
        return self.gamma(e) * zvec + self.beta(e)

class HiveFusion(nn.Module):
    """
    variant:
      - single modality: pass mods=["text"] etc
      - fused: mods=["text","img","vid","aud"]
    """
    def __init__(self, mods, n_obs, n_moods, z=128, multitask=True):
        super().__init__()
        self.mods = mods
        self.z = z
        self.multitask = multitask

        dims = {"text":384, "img":512, "vid":512, "aud":768}
        self.enc = nn.ModuleDict({m: ModEncoder(dims[m], z=z) for m in mods})

        # fuse by concat then project back to z
        self.fuse = nn.Sequential(
            nn.Linear(len(mods)*z, z),
            nn.GELU(),
            nn.LayerNorm(z)
        )

        self.film = FiLM(z=z, n_obs=n_obs)

        # heads
        self.mood_head = nn.Linear(z, n_moods)
        if multitask:
            self.classical_head = nn.Linear(z, 3)
            self.quantum_head   = nn.Linear(z, 3)

    def forward(self, xdict, obs):
        zs = [self.enc[m](xdict[m]) for m in self.mods]
        zcat = torch.cat(zs, dim=-1)
        z = self.fuse(zcat)
        z = self.film(z, obs)
        mood_logits = self.mood_head(z)

        if self.multitask:
            return mood_logits, self.classical_head(z), self.quantum_head(z), z
        return mood_logits, None, None, z

In [75]:
from tqdm import tqdm

@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    total, correct = 0, 0
    loss_sum = 0.0
    for x, obs, mood, classical, quantum in loader:
        x = {k:v.to(device) for k,v in x.items()}
        obs = obs.to(device); mood = mood.to(device)
        classical = classical.to(device); quantum = quantum.to(device)

        logits, c_hat, q_hat, _ = model(x, obs) # Unpack all 4 values, ignoring the latent 'z'
        loss = F.cross_entropy(logits, mood)
        if c_hat is not None:
            loss = loss + 0.25*F.mse_loss(c_hat, classical) + 0.25*F.mse_loss(q_hat, quantum)

        pred = logits.argmax(-1)
        correct += (pred == mood).sum().item()
        total += mood.numel()
        loss_sum += loss.item() * mood.size(0)

    return loss_sum/len(loader.dataset), correct/total

def train(model, train_loader, val_loader, device, epochs=40, lr=3e-4, save_path="hive_best.pt"):
    model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)

    best = 1e9
    for ep in range(1, epochs+1):
        model.train()
        for x, obs, mood, classical, quantum in train_loader:
            x = {k:v.to(device) for k,v in x.items()}
            obs = obs.to(device); mood = mood.to(device)
            classical = classical.to(device); quantum = quantum.to(device)

            opt.zero_grad()
            logits, c_hat, q_hat, _ = model(x, obs) # Unpack all 4 values, ignoring the latent 'z'
            loss = F.cross_entropy(logits, mood)
            if c_hat is not None:
                loss = loss + 0.25*F.mse_loss(c_hat, classical) + 0.25*F.mse_loss(q_hat, quantum)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

        vloss, vacc = evaluate(model, val_loader, device)
        if vloss < best:
            best = vloss
            torch.save(model.state_dict(), save_path)

        if ep % 5 == 0 or ep == 1:
            print(f"ep {ep:02d} | val loss {vloss:.4f} | val acc {vacc:.3f}")

    return best

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

def run_all():
    outdir = "/content/drive/MyDrive" if os.path.exists("/content/drive") else "/content"
    n_obs = len(scripts)
    n_moods = len(moods)

    variants = [
        (["text"], "text_only"),
        (["img"],  "img_only"),
        (["vid"],  "vid_only"),
        (["aud"],  "aud_only"),
        (["text","img","vid","aud"], "fused_all"),
    ]

    results = {}
    for mods, name in variants:
        print("\n===", name, mods, "===")
        model = HiveFusion(mods=mods, n_obs=n_obs, n_moods=n_moods, z=128, multitask=True)
        save_path = os.path.join(outdir, f"hive_{name}.pt")
        best = train(model, train_loader, val_loader, device, epochs=40, lr=3e-4, save_path=save_path)
        vloss, vacc = evaluate(model, val_loader, device)
        results[name] = {"best_val_loss": float(best), "final_val_acc": float(vacc), "ckpt": save_path}
        print("saved:", save_path, "| final val acc:", vacc)

    return results

results = run_all()
results

device: cuda

=== text_only ['text'] ===
ep 01 | val loss 0.6220 | val acc 0.615
ep 05 | val loss 0.3751 | val acc 1.000
ep 10 | val loss 0.0495 | val acc 1.000
ep 15 | val loss 0.0103 | val acc 1.000
ep 20 | val loss 0.0076 | val acc 1.000
ep 25 | val loss 0.0069 | val acc 1.000
ep 30 | val loss 0.0048 | val acc 1.000
ep 35 | val loss 0.0046 | val acc 1.000
ep 40 | val loss 0.0051 | val acc 1.000
saved: /content/drive/MyDrive/hive_text_only.pt | final val acc: 1.0

=== img_only ['img'] ===
ep 01 | val loss 0.7184 | val acc 0.692
ep 05 | val loss 0.5666 | val acc 0.538
ep 10 | val loss 0.5329 | val acc 0.692
ep 15 | val loss 0.5432 | val acc 0.615
ep 20 | val loss 0.5360 | val acc 0.769
ep 25 | val loss 0.5301 | val acc 0.692
ep 30 | val loss 0.4947 | val acc 0.692
ep 35 | val loss 0.6271 | val acc 0.769
ep 40 | val loss 0.7350 | val acc 0.769
saved: /content/drive/MyDrive/hive_img_only.pt | final val acc: 0.7692307692307693

=== vid_only ['vid'] ===
ep 01 | val loss 0.6123 | val acc 0

{'text_only': {'best_val_loss': 0.004144482314586639,
  'final_val_acc': 1.0,
  'ckpt': '/content/drive/MyDrive/hive_text_only.pt'},
 'img_only': {'best_val_loss': 0.48991405963897705,
  'final_val_acc': 0.7692307692307693,
  'ckpt': '/content/drive/MyDrive/hive_img_only.pt'},
 'vid_only': {'best_val_loss': 0.010544471442699432,
  'final_val_acc': 1.0,
  'ckpt': '/content/drive/MyDrive/hive_vid_only.pt'},
 'aud_only': {'best_val_loss': 0.5177057981491089,
  'final_val_acc': 0.6923076923076923,
  'ckpt': '/content/drive/MyDrive/hive_aud_only.pt'},
 'fused_all': {'best_val_loss': 0.005209174007177353,
  'final_val_acc': 1.0,
  'ckpt': '/content/drive/MyDrive/hive_fused_all.pt'}}

In [76]:
!ls -la /content/synth

total 468
drwxr-xr-x 6 root root   4096 Jan 19 18:38 .
drwxr-xr-x 1 root root   4096 Jan 19 20:23 ..
drwxr-xr-x 2 root root   4096 Jan 19 18:37 audio
-rw-r--r-- 1 root root 150656 Jan 19 18:38 E_aud.npy
-rw-r--r-- 1 root root 100480 Jan 19 18:38 E_img.npy
-rw-r--r-- 1 root root  75392 Jan 19 18:38 E_text.npy
-rw-r--r-- 1 root root 100480 Jan 19 18:38 E_vid.npy
drwxr-xr-x 2 root root   4096 Jan 19 18:37 images
-rw-r--r-- 1 root root  20380 Jan 19 18:37 manifest.jsonl
drwxr-xr-x 2 root root   4096 Jan 19 18:37 text
drwxr-xr-x 2 root root   4096 Jan 19 18:37 video


In [77]:
import os, random
import numpy as np
import torch

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


In [78]:
from sklearn.model_selection import StratifiedKFold

N = len(y_mood)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
folds = list(skf.split(np.zeros(N), y_mood))
print("kfold folds:", len(folds), "N:", N)


kfold folds: 5 N: 49


In [79]:
from torch.utils.data import WeightedRandomSampler

def make_balanced_sampler(idx, labels):
    y = labels[idx]
    classes, counts = np.unique(y, return_counts=True)
    count_map = {c:cnt for c,cnt in zip(classes, counts)}
    w = np.array([1.0 / count_map[yy] for yy in y], dtype=np.float64)
    return WeightedRandomSampler(weights=w, num_samples=len(w), replacement=True)


In [80]:
import torch.nn.functional as F

def info_nce(z, mood, temperature=0.2):
    """
    z: (B,Z) latent
    mood: (B,) int
    """
    z = F.normalize(z, dim=-1)
    sim = (z @ z.T) / temperature  # (B,B)

    # mask self
    B = z.size(0)
    self_mask = torch.eye(B, device=z.device).bool()
    sim = sim.masked_fill(self_mask, -1e9)

    # positives: same mood
    pos = (mood.unsqueeze(1) == mood.unsqueeze(0)) & (~self_mask)
    if pos.sum() == 0:
        return z.new_tensor(0.0)

    # log-softmax over rows
    logp = F.log_softmax(sim, dim=1)

    # average log prob of positives for each anchor that has positives
    pos_counts = pos.sum(dim=1)
    valid = pos_counts > 0
    loss = -(logp[pos].view(B, -1).sum(dim=1) / pos_counts.clamp_min(1)).masked_select(valid).mean()
    return loss


In [81]:
import torch.nn.functional as F

def info_nce(z, mood, temperature=0.2):
    """
    z: (B,Z) latent
    mood: (B,) int
    """
    z = F.normalize(z, dim=-1)
    sim = (z @ z.T) / temperature  # (B,B)

    # mask self
    B = z.size(0)
    self_mask = torch.eye(B, device=z.device).bool()
    sim = sim.masked_fill(self_mask, -1e9)

    # positives: same mood
    pos = (mood.unsqueeze(1) == mood.unsqueeze(0)) & (~self_mask)
    if pos.sum() == 0:
        return z.new_tensor(0.0)

    # log-softmax over rows
    logp = F.log_softmax(sim, dim=1)

    # Calculate sum of log probabilities for positive pairs for each anchor
    sum_logp_pos = (logp * pos.float()).sum(dim=1) # (B,)

    # Divide by pos_counts (number of positives per anchor)
    pos_counts = pos.sum(dim=1) # (B,)

    # Take mean only for valid anchors (those with at least one positive)
    valid = pos_counts > 0
    per_anchor_loss = - (sum_logp_pos / pos_counts.clamp_min(1)) # (B,)

    loss = per_anchor_loss.masked_select(valid).mean()
    return loss


In [82]:
# Replace forward in HiveFusion with this version
def forward(self, xdict, obs):
    zs = [self.enc[m](xdict[m]) for m in self.mods]
    zcat = torch.cat(zs, dim=-1)
    z = self.fuse(zcat)
    z = self.film(z, obs)
    mood_logits = self.mood_head(z)

    if self.multitask:
        return mood_logits, self.classical_head(z), self.quantum_head(z), z
    return mood_logits, None, None, z


In [83]:
import torch.nn as nn
from tqdm import tqdm

@torch.no_grad()
def evaluate(model, loader, device, loss_weights):
    model.eval()
    total, correct = 0, 0
    loss_sum = 0.0

    w_cls, w_c, w_q, w_con = loss_weights

    for x, obs, mood, classical, quantum in loader:
        x = {k:v.to(device) for k,v in x.items()}
        obs = obs.to(device); mood = mood.to(device)
        classical = classical.to(device); quantum = quantum.to(device)

        logits, c_hat, q_hat, z = model(x, obs)

        loss = w_cls * F.cross_entropy(logits, mood)
        if c_hat is not None:
            loss = loss + w_c*F.mse_loss(c_hat, classical) + w_q*F.mse_loss(q_hat, quantum)
        if w_con > 0:
            loss = loss + w_con*info_nce(z, mood)

        pred = logits.argmax(-1)
        correct += (pred == mood).sum().item()
        total += mood.numel()
        loss_sum += loss.item() * mood.size(0)

    return loss_sum/len(loader.dataset), correct/total


def train_fold(model, train_loader, val_loader, device,
               epochs=120, lr=3e-4, save_path="hive_best.pt",
               loss_weights=(1.0, 0.25, 0.25, 0.10),
               patience=20):
    """
    loss_weights = (w_cls, w_classical_mse, w_quantum_mse, w_contrastive)
    """
    model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)

    best = 1e9
    bad = 0

    for ep in range(1, epochs+1):
        model.train()
        for x, obs, mood, classical, quantum in train_loader:
            x = {k:v.to(device) for k,v in x.items()}
            obs = obs.to(device); mood = mood.to(device)
            classical = classical.to(device); quantum = quantum.to(device)

            opt.zero_grad()
            logits, c_hat, q_hat, z = model(x, obs)

            w_cls, w_c, w_q, w_con = loss_weights
            loss = w_cls * F.cross_entropy(logits, mood)
            loss = loss + w_c*F.mse_loss(c_hat, classical) + w_q*F.mse_loss(q_hat, quantum)
            if w_con > 0:
                loss = loss + w_con*info_nce(z, mood)

            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

        vloss, vacc = evaluate(model, val_loader, device, loss_weights)

        if vloss < best - 1e-5:
            best = vloss
            bad = 0
            torch.save(model.state_dict(), save_path)
        else:
            bad += 1

        if ep % 10 == 0 or ep == 1:
            print(f"ep {ep:03d} | val loss {vloss:.4f} | val acc {vacc:.3f} | bad {bad}/{patience}")

        if bad >= patience:
            break

    return best


In [84]:
from torch.utils.data import DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

def run_all_kfold():
    outdir = "/content/drive/MyDrive" if os.path.exists("/content/drive") else "/content"
    n_obs = len(scripts)
    n_moods = len(moods)

    variants = [
        (["text"], "text_only"),
        (["img"],  "img_only"),
        (["vid"],  "vid_only"),
        (["aud"],  "aud_only"),
        (["text","img","vid","aud"], "fused_all"),
    ]

    # weights: (CE mood, MSE classical, MSE quantum, contrastive)
    loss_weights = (1.0, 0.25, 0.25, 0.10)

    summary = {}
    for mods, name in variants:
        fold_metrics = []
        print("\n==============================")
        print("VARIANT:", name, mods)

        for fi, (tr_idx, va_idx) in enumerate(folds, start=1):
            # loaders
            train_ds = SynthHive(tr_idx)
            val_ds   = SynthHive(va_idx)

            # optional balancing:
            sampler = make_balanced_sampler(tr_idx, y_mood)
            train_loader = DataLoader(train_ds, batch_size=16, sampler=sampler)
            val_loader   = DataLoader(val_ds, batch_size=16, shuffle=False)

            # model
            model = HiveFusion(mods=mods, n_obs=n_obs, n_moods=n_moods, z=128, multitask=True)

            save_path = os.path.join(outdir, f"hive_{name}_fold{fi}.pt")
            best = train_fold(
                model, train_loader, val_loader, device,
                epochs=200, lr=3e-4,
                save_path=save_path,
                loss_weights=loss_weights,
                patience=25
            )

            vloss, vacc = evaluate(model, val_loader, device, loss_weights)
            fold_metrics.append((float(vloss), float(vacc), save_path))
            print(f"fold {fi}: val_loss={vloss:.4f} val_acc={vacc:.3f} saved={save_path}")

        # aggregate
        losses = [m[0] for m in fold_metrics]
        accs   = [m[1] for m in fold_metrics]
        summary[name] = {
            "mods": mods,
            "mean_val_loss": float(np.mean(losses)),
            "std_val_loss":  float(np.std(losses)),
            "mean_val_acc":  float(np.mean(accs)),
            "std_val_acc":   float(np.std(accs)),
            "checkpoints":   [m[2] for m in fold_metrics],
        }

        print(f"\n{name} | mean acc {summary[name]['mean_val_acc']:.3f} \u00b1 {summary[name]['std_val_acc']:.3f}")

    return summary

summary = run_all_kfold()
summary

device: cuda

VARIANT: text_only ['text']
ep 001 | val loss 1.1581 | val acc 0.800 | bad 0/25
ep 010 | val loss 0.2340 | val acc 1.000 | bad 0/25
ep 020 | val loss 0.1522 | val acc 1.000 | bad 0/25
ep 030 | val loss 0.1476 | val acc 1.000 | bad 2/25
ep 040 | val loss 0.1471 | val acc 1.000 | bad 4/25
ep 050 | val loss 0.1453 | val acc 1.000 | bad 6/25
ep 060 | val loss 0.1440 | val acc 1.000 | bad 0/25
ep 070 | val loss 0.1439 | val acc 1.000 | bad 2/25
ep 080 | val loss 0.1458 | val acc 1.000 | bad 2/25
ep 090 | val loss 0.1440 | val acc 1.000 | bad 12/25
ep 100 | val loss 0.1432 | val acc 1.000 | bad 3/25
ep 110 | val loss 0.1443 | val acc 1.000 | bad 4/25
ep 120 | val loss 0.1442 | val acc 1.000 | bad 14/25
ep 130 | val loss 0.1433 | val acc 1.000 | bad 24/25
ep 140 | val loss 0.1428 | val acc 1.000 | bad 9/25
ep 150 | val loss 0.1432 | val acc 1.000 | bad 6/25
ep 160 | val loss 0.1429 | val acc 1.000 | bad 16/25
ep 170 | val loss 0.1434 | val acc 1.000 | bad 5/25
ep 180 | val loss 

{'text_only': {'mods': ['text'],
  'mean_val_loss': 0.18930740356445314,
  'std_val_loss': 0.10096563851951502,
  'mean_val_acc': 0.9800000000000001,
  'std_val_acc': 0.039999999999999994,
  'checkpoints': ['/content/drive/MyDrive/hive_text_only_fold1.pt',
   '/content/drive/MyDrive/hive_text_only_fold2.pt',
   '/content/drive/MyDrive/hive_text_only_fold3.pt',
   '/content/drive/MyDrive/hive_text_only_fold4.pt',
   '/content/drive/MyDrive/hive_text_only_fold5.pt']},
 'img_only': {'mods': ['img'],
  'mean_val_loss': 0.5787388145923614,
  'std_val_loss': 0.679236866897183,
  'mean_val_acc': 0.8800000000000001,
  'std_val_acc': 0.16,
  'checkpoints': ['/content/drive/MyDrive/hive_img_only_fold1.pt',
   '/content/drive/MyDrive/hive_img_only_fold2.pt',
   '/content/drive/MyDrive/hive_img_only_fold3.pt',
   '/content/drive/MyDrive/hive_img_only_fold4.pt',
   '/content/drive/MyDrive/hive_img_only_fold5.pt']},
 'vid_only': {'mods': ['vid'],
  'mean_val_loss': 0.19761628806591033,
  'std_val_lo

In [85]:
import numpy as np

# --- Quantum: Cirq (works even if certain helper APIs are missing) ---
import cirq

def z_expectations_from_state(psi, n_qubits):
    """Compute <Z_i> for each qubit i from a full statevector psi."""
    # psi: complex statevector length 2^n
    probs = np.abs(psi)**2
    exps = []
    for i in range(n_qubits):
        # Z expectation: sum_{bitstring} (+1 if bit i=0 else -1) * P(bitstring)
        # bit i corresponds to position (n_qubits-1-i) in binary indexing depending on convention;
        # Cirq uses little-endian ordering in many contexts. We'll match that:
        exp = 0.0
        for idx, p in enumerate(probs):
            bit = (idx >> i) & 1  # little-endian
            exp += (1.0 if bit == 0 else -1.0) * p
        exps.append(exp)
    return np.array(exps, dtype=np.float32)

def quantum_encode(features, n_qubits=4, depth=2):
    """
    Map features -> parameterized circuit -> expectation vector in [-1,1]^n_qubits
    """
    qubits = cirq.LineQubit.range(n_qubits)
    circuit = cirq.Circuit()

    # simple feature embedding: Ry rotations on each qubit
    # features assumed length >= n_qubits
    for i, q in enumerate(qubits):
        theta = float(features[i]) * np.pi
        circuit.append(cirq.ry(theta)(q))

    # entangling layers
    for _ in range(depth):
        for i in range(n_qubits - 1):
            circuit.append(cirq.CNOT(qubits[i], qubits[i+1]))
        for i, q in enumerate(qubits):
            circuit.append(cirq.rz(0.25 * np.pi)(q))

    sim = cirq.Simulator()
    result = sim.simulate(circuit)
    psi = np.array(result.final_state_vector, dtype=np.complex64)
    return z_expectations_from_state(psi, n_qubits)

# --- Hive bridge: tiny MLP (replace with your multimodal+observer model later) ---
import torch
import torch.nn as nn

class HiveBridge(nn.Module):
    def __init__(self, d_in, d_q, z_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in + d_q, 128),
            nn.GELU(),
            nn.Linear(128, z_dim),
            nn.LayerNorm(z_dim),
        )
        # translate to Brian2 drive (current) and synapse delta
        self.to_current = nn.Linear(z_dim, 1)      # scalar neuromod drive
        self.to_synapse = nn.Linear(z_dim, 1)      # global synapse gain (demo)

    def forward(self, x, q):
        h = torch.cat([x, q], dim=-1)
        z = self.net(h)
        drive = self.to_current(z)   # (B,1)
        sgain = self.to_synapse(z)   # (B,1)
        return z, drive, sgain

# --- Brian2 substrate ---
from brian2 import *

def run_brian2_hive(drive_scalar, syn_gain, duration_ms=500):
    """
    drive_scalar: float (from Hive) ~ arbitrary
    syn_gain: float (from Hive) ~ arbitrary
    returns: spike times, a simple haptic envelope
    """
    start_scope()

    N = 50
    tau = 10*ms
    eqs = """
    dv/dt = (-v + I)/tau : 1
    I : 1
    """

    G = NeuronGroup(N, eqs, threshold='v>1', reset='v=0', method='euler')
    G.v = 0

    # baseline + Hive drive
    base = 0.6
    drive = float(drive_scalar)
    G.I = base + 0.4*np.tanh(drive)  # keep stable

    # recurrent synapses with Hive gain
    # Declaring 'w' in the Synapses model
    S = Synapses(G, G, model='w : 1', on_pre='v_post += w')
    S.connect(p=0.1)
    S.w = (0.02 + 0.03*np.tanh(float(syn_gain)))  # small excitatory kick

    spikemon = SpikeMonitor(G)
    ratemon = PopulationRateMonitor(G)

    run(duration_ms*ms)

    # "haptic envelope": use population rate (Hz) scaled 0..1
    rate = ratemon.smooth_rate(window='flat', width=20*ms) / Hz
    env = np.clip(rate / 100.0, 0, 1)  # pretend 100 Hz = max vibration
    return spikemon.t/ms, spikemon.i, env

# --- Demo pipeline ---
# input features (your “cognition” vector) – replace with embeddings/sentiment features
x_feat = np.random.randn(8).astype(np.float32)

# quantum vector from first 4 features
q_vec = quantum_encode(x_feat, n_qubits=4, depth=2)

# Hive forward
device = "cuda" if torch.cuda.is_available() else "cpu"
hive = HiveBridge(d_in=8, d_q=4, z_dim=64).to(device).eval()

with torch.no_grad():
    x_t = torch.tensor(x_feat[None, :], device=device)
    q_t = torch.tensor(q_vec[None, :], device=device)
    z, drive, sgain = hive(x_t, q_t)

drive_scalar = float(drive.cpu().numpy().squeeze())
syn_gain = float(sgain.cpu().numpy().squeeze())

# Brian2 run
t_spk, i_spk, haptic_env = run_brian2_hive(drive_scalar, syn_gain, duration_ms=500)

print("Quantum q:", q_vec)
print("Hive drive:", drive_scalar, "syn_gain:", syn_gain)
print("Spikes:", len(t_spk), "Haptic env samples:", len(haptic_env))


INFO       width adjusted from 20. ms to 20.1 ms [brian2.monitors.ratemonitor.adjusted_width]


Quantum q: [ 0.06557437 -0.00461942  0.9071353   0.01032271]
Hive drive: -0.6138136386871338 syn_gain: 0.6879374980926514
Spikes: 0 Haptic env samples: 5000


In [86]:
!pip -q install numpy torch brian2 cirq-core qsimcirq


In [87]:
import torch, torch.nn as nn

class HiveBridge(nn.Module):
    """
    Input: x (your cognition/features), q (quantum expectations)
    Output:
      - z: latent cognition state
      - drive_vec: per-population drive currents (scaled later)
      - syn_gain: global synapse gain (for option 2)
      - gate: neuromodulator/plasticity gate (for option 3)
    """
    def __init__(self, d_in, d_q, z_dim=64, n_pops=3):
        super().__init__()
        self.core = nn.Sequential(
            nn.Linear(d_in + d_q, 128),
            nn.GELU(),
            nn.Linear(128, z_dim),
            nn.LayerNorm(z_dim),
        )
        self.drive = nn.Linear(z_dim, n_pops)  # per-module drive
        self.syn_gain = nn.Linear(z_dim, 1)    # global gain
        self.gate = nn.Linear(z_dim, 1)        # plasticity gate

    def forward(self, x, q):
        z = self.core(torch.cat([x, q], dim=-1))
        drive_vec = self.drive(z)
        syn_gain = self.syn_gain(z)
        gate = self.gate(z)
        return z, drive_vec, syn_gain, gate


In [88]:
from brian2 import *

def run_brian2_drive_only(drive_vec, duration_ms=800, N=60):
    """
    drive_vec: np array shape (3,) from HiveBridge
    Returns: spikes + "haptic envelope" from motor population rate
    """
    start_scope()
    defaultclock.dt = 1*ms

    tau = 10*ms
    eqs = """
    dv/dt = (-v + I)/tau : 1
    I : 1
    """

    # 3 modules: sensory, affective, motor
    Gs = [NeuronGroup(N, eqs, threshold='v>1', reset='v=0', method='euler') for _ in range(3)]
    for G in Gs:
        G.v = 0

    # Map Hive drive -> currents (clipped for stability)
    base = [0.55, 0.55, 0.55]
    drive = np.tanh(np.array(drive_vec, dtype=np.float32))  # [-1,1]
    scales = [0.25, 0.30, 0.35]  # motor slightly more sensitive

    for k, G in enumerate(Gs):
        G.I = base[k] + scales[k] * float(drive[k])

    # Mild recurrent within each pop (fixed)
    for G in Gs:
        S = Synapses(G, G, model='w : 1', on_pre='v_post += w')
        S.connect(p=0.08)
        S.w = 0.02

    spk = [SpikeMonitor(G) for G in Gs]
    rate_motor = PopulationRateMonitor(Gs[2])

    run(duration_ms*ms)

    # Haptic envelope: motor population smoothed rate, normalized
    rate = rate_motor.smooth_rate(window='flat', width=25*ms) / Hz
    env = np.clip(rate / 120.0, 0, 1)  # 120 Hz -> max "vibe"
    return spk, env

In [89]:
def run_brian2_drive_plus_syn_gain(drive_vec, syn_gain, duration_ms=800, N=60):
    start_scope()
    defaultclock.dt = 1*ms

    tau = 10*ms
    eqs = """
    dv/dt = (-v + I)/tau : 1
    I : 1
    """

    Gs = [NeuronGroup(N, eqs, threshold='v>1', reset='v=0', method='euler') for _ in range(3)]
    for G in Gs:
        G.v = 0

    base = [0.55, 0.55, 0.55]
    drive = np.tanh(np.array(drive_vec, dtype=np.float32))
    scales = [0.25, 0.30, 0.35]
    for k, G in enumerate(Gs):
        G.I = base[k] + scales[k] * float(drive[k])

    # synapse gain in a safe range
    g = float(np.tanh(float(syn_gain)))  # [-1,1]
    gain = 0.02 + 0.03 * (g + 1)/2.0     # ~ [0.02, 0.05]

    # Intra-module
    for G in Gs:
        S = Synapses(G, G, model='w : 1', on_pre='v_post += w')
        S.connect(p=0.08)
        S.w = 0.02

    # Inter-module: sensory->affective->motor (translated by Hive syn_gain)
    Sa = Synapses(Gs[0], Gs[1], model='w : 1', on_pre='v_post += w'); Sa.connect(p=0.10); Sa.w = gain
    Am = Synapses(Gs[1], Gs[2], model='w : 1', on_pre='v_post += w'); Am.connect(p=0.12); Am.w = gain

    spk = [SpikeMonitor(G) for G in Gs]
    rate_motor = PopulationRateMonitor(Gs[2])

    run(duration_ms*ms)

    rate = rate_motor.smooth_rate(window='flat', width=25*ms) / Hz
    env = np.clip(rate / 120.0, 0, 1)
    return spk, env, gain

In [90]:
def run_brian2_drive_plus_gated_stdp(drive_vec, gate, duration_ms=2000, N=60):
    start_scope()
    defaultclock.dt = 1*ms

    tau = 10*ms
    eqs = """
    dv/dt = (-v + I)/tau : 1
    I : 1
    """

    # 3 modules
    Gs = [NeuronGroup(N, eqs, threshold='v>1', reset='v=0', method='euler') for _ in range(3)]
    for G in Gs:
        G.v = 0

    base = [0.55, 0.55, 0.55]
    drive = np.tanh(np.array(drive_vec, dtype=np.float32))
    scales = [0.25, 0.30, 0.35]
    for k, G in enumerate(Gs):
        G.I = base[k] + scales[k] * float(drive[k])

    # neuromodulator gate in [0,1]
    g = float(1 / (1 + np.exp(-float(gate))))  # sigmoid

    # STDP parameters
    tau_pre = 20*ms
    tau_post = 20*ms
    A_pre = 0.01 * g
    A_post = -0.012 * g
    wmin, wmax = 0.0, 0.08

    # Plastic synapses: affective -> motor
    S = Synapses(
        Gs[1], Gs[2],
        model="""
        w : 1
        dpre/dt = -pre/tau_pre : 1 (event-driven)
        dpost/dt = -post/tau_post : 1 (event-driven)
        """,
        on_pre="""
        v_post += w
        pre += A_pre
        w = clip(w + post, wmin, wmax)
        """,
        on_post="""
        post += A_post
        w = clip(w + pre, wmin, wmax)
        """,
        method='euler'
    )
    S.connect(p=0.12)
    S.w = 0.02

    # Fixed sensory->affective so sensory can stimulate learning downstream
    Sa = Synapses(Gs[0], Gs[1], model='w : 1', on_pre='v_post += w')
    Sa.connect(p=0.10)
    Sa.w = 0.03

    spk = [SpikeMonitor(G) for G in Gs]
    rate_motor = PopulationRateMonitor(Gs[2])

    run(duration_ms*ms)

    rate = rate_motor.smooth_rate(window='flat', width=30*ms) / Hz
    env = np.clip(rate / 120.0, 0, 1)
    return spk, env, g, float(np.mean(S.w))

In [91]:
import numpy as np
import torch

def run_pipeline(x_feat, n_qubits=4):
    # quantum
    q = quantum_encode(x_feat, n_qubits=n_qubits, depth=2)

    # hive
    device = "cuda" if torch.cuda.is_available() else "cpu"
    hive = HiveBridge(d_in=len(x_feat), d_q=n_qubits, z_dim=64, n_pops=3).to(device).eval()

    with torch.no_grad():
        x_t = torch.tensor(x_feat[None, :], device=device, dtype=torch.float32)
        q_t = torch.tensor(q[None, :], device=device, dtype=torch.float32)
        z, drive_vec, syn_gain, gate = hive(x_t, q_t)

    drive_vec = drive_vec.cpu().numpy().squeeze()
    syn_gain = float(syn_gain.cpu().numpy().squeeze())
    gate = float(gate.cpu().numpy().squeeze())

    # Brian2 option 1
    spk1, env1 = run_brian2_drive_only(drive_vec)

    # Brian2 option 2
    spk2, env2, gain_used = run_brian2_drive_plus_syn_gain(drive_vec, syn_gain)

    # Brian2 option 3
    spk3, env3, gate_used, mean_w = run_brian2_drive_plus_gated_stdp(drive_vec, gate)

    return {
        "q": q,
        "drive_vec": drive_vec,
        "syn_gain_raw": syn_gain,
        "syn_gain_used": gain_used,
        "gate_raw": gate,
        "gate_used": gate_used,
        "mean_plastic_w": mean_w,
        "env1": env1,
        "env2": env2,
        "env3": env3,
        "spk1": spk1,
        "spk2": spk2,
        "spk3": spk3,
    }

# demo input (replace with your fused embeddings / sentiment vector slice)
x_feat = np.random.randn(8).astype(np.float32)
out = run_pipeline(x_feat)
print("q:", out["q"])
print("drive_vec:", out["drive_vec"])
print("syn_gain_used:", out["syn_gain_used"], "gate_used:", out["gate_used"], "mean_plastic_w:", out["mean_plastic_w"])

WARNING    The object 'synapses' is getting deleted, but was never included in a network. This probably means that you did not store the object reference in a variable, or that the variable was not used to construct the network.
The object was created here (most recent call only):
  File '/tmp/ipython-input-2872369708.py', line 32, in run_brian2_drive_only
    S = Synapses(G, G, model='w : 1', on_pre='v_post += w') [brian2.core.base.unused_brian_object]
WARNING    The object 'synapses_1' is getting deleted, but was never included in a network. This probably means that you did not store the object reference in a variable, or that the variable was not used to construct the network.
The object was created here (most recent call only):
  File '/tmp/ipython-input-2872369708.py', line 32, in run_brian2_drive_only
    S = Synapses(G, G, model='w : 1', on_pre='v_post += w') [brian2.core.base.unused_brian_object]
WARNING    The object 'synapses' is getting deleted, but was never included in a n

q: [-0.81128997  0.04959289  0.9672118   0.16742599]
drive_vec: [0.6698644  0.7354847  0.10315648]
syn_gain_used: 0.038914656875971586 gate_used: 0.6394589718590538 mean_plastic_w: 0.019999999999999993


In [92]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm

class ZProject(nn.Module):
    """Project student z -> teacher z dim so we can match features."""
    def __init__(self, z_s, z_t):
        super().__init__()
        self.proj = nn.Linear(z_s, z_t)
    def forward(self, z_s):
        return self.proj(z_s)

def distill_train(
    teacher, student, train_loader, val_loader, device,
    epochs=20, lr=3e-4,
    T=2.0,                 # temperature for soft labels
    w_hard=1.0,            # weight for hard label loss
    w_soft=0.5,            # weight for soft KL
    w_reg=0.25,            # weight for classical/quantum regressions (if enabled)
    w_z=0.2,               # weight for feature distill
    save_path="student_distilled.pt"
):
    teacher = teacher.to(device).eval()
    student = student.to(device).train()

    # infer z dims once (quickly) by a single batch
    x0, obs0, mood0, classical0, quantum0 = next(iter(train_loader))
    x0 = {k:v.to(device) for k,v in x0.items()}
    obs0 = obs0.to(device)
    with torch.no_grad():
        _, _, _, zt = teacher(x0, obs0)
        _, _, _, zs = student(x0, obs0)
    zproj = ZProject(z_s=zs.shape[-1], z_t=zt.shape[-1]).to(device)

    opt = torch.optim.AdamW(list(student.parameters()) + list(zproj.parameters()),
                            lr=lr, weight_decay=1e-2)

    best = 1e9
    for ep in range(1, epochs + 1):
        student.train()
        zproj.train()
        loss_sum = 0.0

        for x, obs, mood, classical, quantum in tqdm(train_loader, desc=f"distill ep {ep}", leave=False):
            x = {k:v.to(device) for k,v in x.items()}
            obs = obs.to(device)
            mood = mood.to(device)
            classical = classical.to(device)
            quantum = quantum.to(device)

            with torch.no_grad():
                t_logits, t_c, t_q, t_z = teacher(x, obs)

            s_logits, s_c, s_q, s_z = student(x, obs)

            # 1) hard supervised loss
            hard = F.cross_entropy(s_logits, mood)

            # 2) soft distillation loss (KL between teacher/student mood distributions)
            # KL(student || teacher) or KL(teacher || student) both ok; common is student vs teacher
            t_probs = F.softmax(t_logits / T, dim=-1)
            s_logp  = F.log_softmax(s_logits / T, dim=-1)
            soft = F.kl_div(s_logp, t_probs, reduction="batchmean") * (T * T)

            # 3) regression (if your model returns these)
            reg = 0.0
            if (s_c is not None) and (s_q is not None) and (t_c is not None) and (t_q is not None):
                # keep student aligned with ground-truth, optionally also match teacher
                reg = F.mse_loss(s_c, classical) + F.mse_loss(s_q, quantum)

            # 4) feature distillation: match latent z
            z_loss = F.mse_loss(zproj(s_z), t_z)

            loss = w_hard*hard + w_soft*soft + w_reg*reg + w_z*z_loss

            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(student.parameters(), 1.0)
            opt.step()

            loss_sum += loss.item() * mood.size(0)

        train_loss = loss_sum / len(train_loader.dataset)

        # quick val (hard-only) to choose best student
        student.eval()
        zproj.eval()
        vloss, vacc = 0.0, 0.0
        total, correct = 0, 0
        with torch.no_grad():
            for x, obs, mood, classical, quantum in val_loader:
                x = {k:v.to(device) for k,v in x.items()}
                obs = obs.to(device)
                mood = mood.to(device)
                classical = classical.to(device)
                quantum = quantum.to(device)
                logits, c_hat, q_hat, _ = student(x, obs)
                loss = F.cross_entropy(logits, mood)
                if c_hat is not None and q_hat is not None:
                    loss = loss + 0.25*F.mse_loss(c_hat, classical) + 0.25*F.mse_loss(q_hat, quantum)
                pred = logits.argmax(-1)
                correct += (pred == mood).sum().item()
                total += mood.numel()
                vloss += loss.item() * mood.size(0)

        vloss = vloss / len(val_loader.dataset)
        vacc = correct / max(1, total)

        print(f"ep {ep:02d} train={train_loss:.4f} val={vloss:.4f} val_acc={vacc:.3f}")

        if vloss < best:
            best = vloss
            torch.save({"student": student.state_dict(), "zproj": zproj.state_dict()}, save_path)
            print("  saved:", save_path)

    return save_path


In [92]:
import neuralink
import numpy as np

# Hive Synchrony Data extracted from the Canvas
data_str = """3157,0,0.998,0.814
3158,0,0.998,0.824
3159,0,0.999,0.844
3160,0,0.996,0.832
3161,0,0.999,0.849
3162,0,0.999,0.858
3163,0,0.999,0.885
3164,0,0.998,0.840
3165,0,0.998,0.784
3166,0,0.998,0.851
3167,0,0.998,0.819
3168,0,0.998,0.878
3169,0,0.997,0.826
3170,0,0.999,0.841
3171,0,0.999,0.823
3172,0,0.997,0.831
3173,0,0.998,0.816
3174,0,0.998,0.850
3175,0,0.998,0.839
3176,0,0.998,0.835
3177,0,0.998,0.817
3178,0,0.999,0.835
3179,0,0.998,0.817
3180,0,0.999,0.895
3181,0,0.999,0.817
3182,0,0.999,0.839
3183,0,0.999,0.790
3184,0,0.999,0.766
3185,0,0.998,0.878
3186,0,0.998,0.797
3187,0,0.998,0.821
3188,0,0.998,0.871
3189,0,0.999,0.863
3190,0,0.999,0.862
3191,0,0.999,0.829
3192,0,0.997,0.869
3193,0,0.999,0.874
3194,0,0.997,0.799
3195,0,0.999,0.897
3196,0,0.998,0.769
3197,0,0.997,0.790
3198,0,0.999,0.865
3199,0,0.999,0.832
3200,0,0.999,0.898
3201,0,0.998,0.851
3202,0,0.999,0.848
3203,0,0.999,0.868
3204,0,0.999,0.867
3205,0,0.999,0.819
3206,0,0.999,0.811
3207,0,0.999,0.856
3208,0,0.998,0.846
3209,0,0.998,0.828
3210,0,0.999,0.867
3211,0,0.999,0.884
3212,0,0.998,0.847
3213,0,0.998,0.806
3214,0,0.999,0.836
3215,0,0.999,0.853
3216,0,0.999,0.823
3217,0,0.999,0.863
3218,0,0.999,0.859
3219,0,0.998,0.850
3220,0,0.999,0.842
3221,0,0.999,0.837
3222,0,0.999,0.849
3223,0,0.998,0.850
3224,0,0.997,0.788
3225,0,0.999,0.858
3226,0,0.998,0.822
3227,0,0.998,0.829
3228,0,0.999,0.851
3229,0,0.999,0.833
3230,0,0.998,0.846
3231,0,0.999,0.825
3232,0,0.998,0.860
3233,0,0.999,0.842
3234,0,0.999,0.864
3235,0,0.998,0.814
3236,0,0.998,0.841
3237,0,0.998,0.849
3238,0,0.999,0.895
3239,0,0.998,0.839
3240,0,0.999,0.918
3241,0,0.999,0.854
3242,0,0.999,0.861
3243,0,0.998,0.812
3244,0,0.999,0.864
3245,0,0.999,0.814
3246,0,0.999,0.858
3247,0,0.999,0.872
3248,0,0.997,0.823
3249,0,0.999,0.861
3250,0,0.999,0.867
3251,0,0.999,0.833
3252,0,0.999,0.801
3253,0,0.999,0.901
3254,0,0.998,0.845
3255,0,0.998,0.808
3256,0,0.999,0.821
3257,0,0.997,0.818
3258,0,0.999,0.797
3259,0,0.999,0.859
3260,0,0.998,0.796
3261,0,0.998,0.794
3262,0,0.999,0.862
3263,0,0.999,0.852
3264,0,0.997,0.902
3265,0,0.999,0.824
3266,0,0.999,0.832
3267,0,0.999,0.883
3268,0,0.999,0.825
3269,0,0.998,0.854
3270,0,0.998,0.881
3271,0,0.999,0.793
3272,0,0.999,0.838
3273,0,0.999,0.868
3274,0,0.999,0.873
3275,0,0.998,0.818
3276,0,0.998,0.840
3277,0,0.999,0.880
3278,0,0.999,0.875
3279,0,0.999,0.853
3280,0,0.999,0.848
3281,0,0.999,0.854
3282,0,0.996,0.827
3283,0,0.999,0.804
3284,0,0.998,0.872
3285,0,0.997,0.775"""

class HivePerceptron:
    def __init__(self, learning_rate=0.01, epochs=100):
        self.lr = learning_rate
        self.epochs = epochs
        self.weights = None
        self.bias = 0

    def fit(self, X, y):
        # Initialize weights for [Synchrony, Haptic_Resonance]
        self.weights = np.zeros(X.shape[1])
        self.bias = 0

        for _ in range(self.epochs):
            for idx, x_i in enumerate(X):
                linear_output = np.dot(x_i, self.weights) + self.bias
                y_predicted = 1 if linear_output >= 0.5 else 0 # Threshold for Hive Event

                # Update rule
                update = self.lr * (y[idx] - y_predicted)
                self.weights += update * x_i
                self.bias += update

    def predict(self, X):
        linear_output = np.dot(X, self.weights) + self.bias
        return [1 if i >= 0.5 else 0 for i in linear_output]

# 1. Prepare Data
lines = data_str.strip().split('\n')
X = []
y = []

# Since the data provided is 100% "HIGH SYNCHRONY EVENT" (y=1),
# we'll train it to recognize this specific pattern.
for line in lines:
    parts = line.split(',')
    # Features: Synchrony (idx 2), Haptic Resonance (idx 3)
    X.append([float(parts[2]), float(parts[3])])
    y.append(1) # Label for HIGH SYNCHRONY EVENT

X = np.array(X)
y = np.array(y)

# 2. Train Model
model = HivePerceptron(learning_rate=0.1, epochs=50)
model.fit(X, y)

print("Training Complete.")
print(f"Learned Weights: {model.weights}")
print(f"Learned Bias: {model.bias}")

# 3. Test with a hypothetical 'Low' state
test_data = np.array([
    [0.999, 0.850], # Should be high (similar to data)
    [0.400, 0.300]  # Should be low (synthetic)
])

predictions = model.predict(test_data)
print("\nTesting Model:")
for i, pred in enumerate(predictions):
    status = "HIGH SYNCHRONY" if pred == 1 else "STABLE/LOW"
    print(f"Inputs {test_data[i]} -> Prediction: {status}")

In [93]:
!nvidia-smi
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
device


Mon Jan 19 20:36:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P0             27W /   70W |    3776MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

'cuda'

In [94]:
!pip -q install --upgrade pip
!pip -q install "transformers>=4.38" accelerate safetensors einops
!pip -q install sentence-transformers
!pip -q install open-clip-torch pillow
!pip -q install librosa soundfile


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 28.7 MB/s eta 0:00:00


In [95]:
!pip -q install decord av


In [96]:
TEXT_MODEL  = "intfloat/e5-base-v2"               # text embeddings
AUDIO_MODEL = "facebook/wav2vec2-base-960h"       # audio embeddings
# OpenCLIP uses public checkpoints; no key needed:
IMAGE_MODEL = ("ViT-B-32", "laion2b_s34b_b79k")   # image embeddings


In [97]:
import numpy as np
import torch

# ---- Text ----
from sentence_transformers import SentenceTransformer
text_encoder = SentenceTransformer(TEXT_MODEL, device=device)

# ---- Image (OpenCLIP) ----
import open_clip
from PIL import Image
image_encoder, _, image_preprocess = open_clip.create_model_and_transforms(
    IMAGE_MODEL[0], pretrained=IMAGE_MODEL[1], device=device
)
image_encoder.eval()

# ---- Audio ----
from transformers import AutoProcessor, AutoModel
audio_processor = AutoProcessor.from_pretrained(AUDIO_MODEL)
audio_encoder = AutoModel.from_pretrained(AUDIO_MODEL).to(device).eval()

print("Loaded:", TEXT_MODEL, IMAGE_MODEL, AUDIO_MODEL)


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/650 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

...

vocab.json:   0%|          | 0.00/291 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

Some weights of Wav2Vec2Model were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Loaded: intfloat/e5-base-v2 ('ViT-B-32', 'laion2b_s34b_b79k') facebook/wav2vec2-base-960h


In [98]:
import librosa

def embed_text(texts, batch_size=32, normalize=True):
    # For E5 models, prefixing helps: "query: ..." / "passage: ..."
    # If your texts are “content”, use passage:
    texts2 = [("passage: " + t) for t in texts]
    emb = text_encoder.encode(
        texts2, batch_size=batch_size,
        convert_to_numpy=True, normalize_embeddings=normalize,
        show_progress_bar=True
    )
    return emb.astype(np.float32)

@torch.no_grad()
def embed_images(image_paths, batch_size=32, normalize=True):
    vecs = []
    for i in range(0, len(image_paths), batch_size):
        batch = image_paths[i:i+batch_size]
        imgs = [image_preprocess(Image.open(p).convert("RGB")) for p in batch]
        x = torch.stack(imgs).to(device)
        z = image_encoder.encode_image(x)
        if normalize:
            z = z / z.norm(dim=-1, keepdim=True).clamp_min(1e-6)
        vecs.append(z.detach().cpu().numpy().astype(np.float32))
    return np.concatenate(vecs, axis=0)

@torch.no_grad()
def embed_audio(audio_paths, target_sr=16000, batch_size=8, normalize=True):
    vecs = []
    for i in range(0, len(audio_paths), batch_size):
        batch = audio_paths[i:i+batch_size]
        waves = []
        for p in batch:
            w, _ = librosa.load(p, sr=target_sr, mono=True)
            waves.append(w)

        inputs = audio_processor(waves, sampling_rate=target_sr, return_tensors="pt", padding=True)
        inputs = {k:v.to(device) for k,v in inputs.items()}
        out = audio_encoder(**inputs)
        z = out.last_hidden_state.mean(dim=1)  # (B,H)
        if normalize:
            z = z / z.norm(dim=-1, keepdim=True).clamp_min(1e-6)
        vecs.append(z.detach().cpu().numpy().astype(np.float32))
    return np.concatenate(vecs, axis=0)


In [99]:
import pandas as pd
from pathlib import Path

# The synthetic data is generated directly in /content/synth
# and the manifest.jsonl and .npy embeddings are already created and loaded.
# The `records` variable from cell bydqcFAki_ev contains the parsed manifest data.
# The `E_text`, `E_img`, `E_aud`, `E_vid` arrays are already loaded.

# Create a DataFrame from the `records` list (which contains parsed manifest.jsonl data)
df = pd.DataFrame(records)

# Adjust column names based on the keys available in `records`
# The actual text content is in the 'text' column of the dataframe
TEXT_COL  = "text"
IMG_COL   = "image_path"
AUD_COL   = "audio_path"
MOOD_COL  = "mood"  # The 'mood' extracted from text is directly in this column

# 'classical' and 'quantum' are stored as numpy arrays within the DataFrame cells
CLASS_COL = "classical"
QUANT_COL = "quantum"

# 'script_id' from records corresponds to the observer ID
OBS_COLS = [c for c in df.columns if c.startswith("script_id")] # 'script_id' is directly available

# The original text and paths are now accessible directly from df
texts    = df[TEXT_COL].tolist()
img_paths = df[IMG_COL].tolist()
aud_paths = df[AUD_COL].tolist()

print("samples:", len(df), "obs_cols:", OBS_COLS)
print("First 5 rows of the DataFrame:")
print(df.head())

samples: 49 obs_cols: ['script_id']
First 5 rows of the DataFrame:
   i                   timestamp script_name  script_id  cycle  \
0  0  2025-08-17T09:56:43.222236       model          4      1   
1  1  2025-08-17T09:56:43.749270      cookie          3      1   
2  2  2025-08-17T09:56:44.239768      client          2      1   
3  3  2025-08-17T09:56:45.642759       model          4      1   
4  4  2025-08-17T09:56:45.643461       model          4      2   

                                                text  \
0  [model] cycle=1 mood=drifting. classical(H=0.7...   
1  [cookie] cycle=1 mood=drifting. classical(H=0....   
2  [client] cycle=1 mood=drifting. classical(H=0....   
3  [model] cycle=1 mood=drifting. classical(H=0.7...   
4  [model] cycle=2 mood=drifting. classical(H=0.7...   

                                         image_path  \
0   /content/synth/images/model_c000001_0000000.png   
1  /content/synth/images/cookie_c000001_0000001.png   
2  /content/synth/images/client_c0

In [100]:
# Define the teacher model (fused_all variant from previous k-fold training)
teacher_mods = ["text", "img", "vid", "aud"]
teacher_model = HiveFusion(mods=teacher_mods, n_obs=n_obs, n_moods=n_moods, z=128, multitask=True)
teacher_ckpt_path = summary["fused_all"]["checkpoints"][0] # Using the checkpoint from the first fold
teacher_model.load_state_dict(torch.load(teacher_ckpt_path, map_location=device))
teacher_model = teacher_model.to(device).eval()

# Define the student model (e.g., text_only or aud_only, as they performed less well but are simpler)
student_mods = ["aud"] # Let's try to distill knowledge into the audio-only model
student_model = HiveFusion(mods=student_mods, n_obs=n_obs, n_moods=n_moods, z=128, multitask=True)
student_model = student_model.to(device).train()

print("Teacher model loaded from:", teacher_ckpt_path)
print("Student model defined for modalities:", student_mods)

Teacher model loaded from: /content/drive/MyDrive/hive_fused_all_fold1.pt
Student model defined for modalities: ['aud']


In [101]:
# Run knowledge distillation
save_distilled_path = "/content/drive/MyDrive/hive_aud_distilled.pt" if os.path.exists("/content/drive") else "/content/hive_aud_distilled.pt"

distill_train(
    teacher=teacher_model,
    student=student_model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    epochs=100,
    lr=3e-4,
    T=2.0,
    w_hard=1.0,
    w_soft=0.5,
    w_reg=0.25,
    w_z=0.2,
    save_path=save_distilled_path
)

print("Distillation training complete. Distilled student model saved to:", save_distilled_path)

ep 01 train=2.8142 val=0.7317 val_acc=0.692
  saved: /content/drive/MyDrive/hive_aud_distilled.pt


ep 02 train=2.4154 val=0.6344 val_acc=0.692
  saved: /content/drive/MyDrive/hive_aud_distilled.pt


ep 03 train=2.1894 val=0.5933 val_acc=0.692
  saved: /content/drive/MyDrive/hive_aud_distilled.pt


ep 04 train=2.0907 val=0.5819 val_acc=0.538
  saved: /content/drive/MyDrive/hive_aud_distilled.pt


ep 05 train=2.0485 val=0.5813 val_acc=0.538
  saved: /content/drive/MyDrive/hive_aud_distilled.pt


...

ep 96 train=1.2300 val=1.1749 val_acc=0.538


ep 97 train=1.1208 val=1.0640 val_acc=0.538


ep 98 train=1.1574 val=1.0287 val_acc=0.846


ep 99 train=1.0867 val=1.0524 val_acc=0.538


ep 100 train=1.0940 val=1.0439 val_acc=0.538
Distillation training complete. Distilled student model saved to: /content/drive/MyDrive/hive_aud_distilled.pt


The knowledge distillation process aims to transfer the capabilities of a larger, more complex 'teacher' model (in this case, the `fused_all` multimodal model) to a smaller, more efficient 'student' model (here, an `aud_only` model). This is done by training the student not only on the ground-truth labels (hard labels) but also on the softened predictions and latent representations of the teacher model (soft labels and feature distillation).

After the distillation training, the `aud_only` student model should ideally achieve better performance than it would have if trained from scratch, especially in areas where the teacher had strong performance due to other modalities.

In [102]:
print("DataFrame head:")
display(df.head())

print("DataFrame info:")
df.info()

DataFrame head:


,i,timestamp,script_name,script_id,cycle,text,image_path,audio_path,video_path
0,0,2025-08-17T09:56:43.222236,model,4,1,[model] cycle=1 mood=drifting. classical(H=0.7...,/content/synth/images/model_c000001_0000000.png,/content/synth/audio/model_c000001_0000000.wav,/content/synth/video/model_c000001_0000000.mp4
1,1,2025-08-17T09:56:43.749270,cookie,3,1,[cookie] cycle=1 mood=drifting. classical(H=0....,/content/synth/images/cookie_c000001_0000001.png,/content/synth/audio/cookie_c000001_0000001.wav,/content/synth/video/cookie_c000001_0000001.mp4
2,2,2025-08-17T09:56:44.239768,client,2,1,[client] cycle=1 mood=drifting. classical(H=0....,/content/synth/images/client_c000001_0000002.png,/content/synth/audio/client_c000001_0000002.wav,/content/synth/video/client_c000001_0000002.mp4
3,3,2025-08-17T09:56:45.642759,model,4,1,[model] cycle=1 mood=drifting. classical(H=0.7...,/content/synth/images/model_c000001_0000003.png,/content/synth/audio/model_c000001_0000003.wav,/content/synth/video/model_c000001_0000003.mp4
4,4,2025-08-17T09:56:45.643461,model,4,2,[model] cycle=2 mood=drifting. classical(H=0.7...,/content/synth/images/model_c000002_0000004.png,/content/synth/audio/model_c000002_0000004.wav,/content/synth/video/model_c000002_0000004.mp4


DataFrame info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49 entries, 0 to 48
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   i            49 non-null     int64 
 1   timestamp    49 non-null     object
 2   script_name  49 non-null     object
 3   script_id    49 non-null     int64 
 4   cycle        49 non-null     int64 
 5   text         49 non-null     object
 6   image_path   49 non-null     object
 7   audio_path   49 non-null     object
 8   video_path   49 non-null     object
dtypes: int64(3), object(6)
memory usage: 3.6+ KB


In [104]:
E_text = embed_text(texts, batch_size=32)
E_img  = embed_images(img_paths, batch_size=32)
E_aud  = embed_audio(aud_paths, batch_size=8)

print(E_text.shape, E_img.shape, E_aud.shape)

CACHE = Path(ROOT) / "emb_cache"
CACHE.mkdir(parents=True, exist_ok=True)

np.save(CACHE / "E_text.npy", E_text)
np.save(CACHE / "E_img.npy",  E_img)
np.save(CACHE / "E_aud.npy",  E_aud)

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

(49, 768) (49, 512) (49, 768)


In [105]:
import brian2
import cirq
import numpy as np

# Hive Synchrony Data extracted from the Canvas
data_str = """3157,0,0.998,0.814
3158,0,0.998,0.824
3159,0,0.999,0.844
3160,0,0.996,0.832
3161,0,0.999,0.849
3162,0,0.999,0.858
3163,0,0.999,0.885
3164,0,0.998,0.840
3165,0,0.998,0.784
3166,0,0.998,0.851
3167,0,0.998,0.819
3168,0,0.998,0.878
3169,0,0.997,0.826
3170,0,0.999,0.841
3171,0,0.999,0.823
3172,0,0.997,0.831
3173,0,0.998,0.816
3174,0,0.998,0.850
3175,0,0.998,0.839
3176,0,0.998,0.835
3177,0,0.998,0.817
3178,0,0.999,0.835
3179,0,0.998,0.817
3180,0,0.999,0.895
3181,0,0.999,0.817
3182,0,0.999,0.839
3183,0,0.999,0.790
3184,0,0.999,0.766
3185,0,0.998,0.878
3186,0,0.998,0.797
3187,0,0.998,0.821
3188,0,0.998,0.871
3189,0,0.999,0.863
3190,0,0.999,0.862
3191,0,0.999,0.829
3192,0,0.997,0.869
3193,0,0.999,0.874
3194,0,0.997,0.799
3195,0,0.999,0.897
3196,0,0.998,0.769
3197,0,0.997,0.790
3198,0,0.999,0.865
3199,0,0.999,0.832
3200,0,0.999,0.898
3201,0,0.998,0.851
3202,0,0.999,0.848
3203,0,0.999,0.868
3204,0,0.999,0.867
3205,0,0.999,0.819
3206,0,0.999,0.811
3207,0,0.999,0.856
3208,0,0.998,0.846
3209,0,0.998,0.828
3210,0,0.999,0.867
3211,0,0.999,0.884
3212,0,0.998,0.847
3213,0,0.998,0.806
3214,0,0.999,0.836
3215,0,0.999,0.853
3216,0,0.999,0.823
3217,0,0.999,0.863
3218,0,0.999,0.859
3219,0,0.998,0.850
3220,0,0.999,0.842
3221,0,0.999,0.837
3222,0,0.999,0.849
3223,0,0.998,0.850
3224,0,0.997,0.788
3225,0,0.999,0.858
3226,0,0.998,0.822
3227,0,0.998,0.829
3228,0,0.999,0.851
3229,0,0.999,0.833
3230,0,0.998,0.846
3231,0,0.999,0.825
3232,0,0.998,0.860
3233,0,0.999,0.842
3234,0,0.999,0.864
3235,0,0.998,0.814
3236,0,0.998,0.841
3237,0,0.998,0.849
3238,0,0.999,0.895
3239,0,0.998,0.839
3240,0,0.999,0.918
3241,0,0.999,0.854
3242,0,0.999,0.861
3243,0,0.998,0.812
3244,0,0.999,0.864
3245,0,0.999,0.814
3246,0,0.999,0.858
3247,0,0.999,0.872
3248,0,0.997,0.823
3249,0,0.999,0.861
3250,0,0.999,0.867
3251,0,0.999,0.833
3252,0,0.999,0.801
3253,0,0.999,0.901
3254,0,0.998,0.845
3255,0,0.998,0.808
3256,0,0.999,0.821
3257,0,0.997,0.818
3258,0,0.999,0.797
3259,0,0.999,0.859
3260,0,0.998,0.796
3261,0,0.998,0.794
3262,0,0.999,0.862
3263,0,0.999,0.852
3264,0,0.997,0.902
3265,0,0.999,0.824
3266,0,0.999,0.832
3267,0,0.999,0.883
3268,0,0.999,0.825
3269,0,0.998,0.854
3270,0,0.998,0.881
3271,0,0.999,0.793
3272,0,0.999,0.838
3273,0,0.999,0.868
3274,0,0.999,0.873
3275,0,0.998,0.818
3276,0,0.998,0.840
3277,0,0.999,0.880
3278,0,0.999,0.875
3279,0,0.999,0.853
3280,0,0.999,0.848
3281,0,0.999,0.854
3282,0,0.996,0.827
3283,0,0.999,0.804
3284,0,0.998,0.872
3285,0,0.997,0.775"""

class HivePerceptron:
    def __init__(self, learning_rate=0.01, epochs=100):
        self.lr = learning_rate
        self.epochs = epochs
        self.weights = None
        self.bias = 0

    def fit(self, X, y):
        # Initialize weights for [Synchrony, Haptic_Resonance]
        self.weights = np.zeros(X.shape[1])
        self.bias = 0

        for _ in range(self.epochs):
            for idx, x_i in enumerate(X):
                linear_output = np.dot(x_i, self.weights) + self.bias
                y_predicted = 1 if linear_output >= 0.5 else 0 # Threshold for Hive Event

                # Update rule
                update = self.lr * (y[idx] - y_predicted)
                self.weights += update * x_i
                self.bias += update

    def predict(self, X):
        linear_output = np.dot(X, self.weights) + self.bias
        return [1 if i >= 0.5 else 0 for i in linear_output]

# 1. Prepare Data
lines = data_str.strip().split('\n')
X = []
y = []

# Since the data provided is 100% "HIGH SYNCHRONY EVENT" (y=1),
# we'll train it to recognize this specific pattern.
for line in lines:
    parts = line.split(',')
    # Features: Synchrony (idx 2), Haptic Resonance (idx 3)
    X.append([float(parts[2]), float(parts[3])])
    y.append(1) # Label for HIGH SYNCHRONY EVENT

X = np.array(X)
y = np.array(y)

# 2. Train Model
model = HivePerceptron(learning_rate=0.1, epochs=50)
model.fit(X, y)

print("Training Complete.")
print(f"Learned Weights: {model.weights}")
print(f"Learned Bias: {model.bias}")

# 3. Test with a hypothetical 'Low' state
test_data = np.array([
    [0.999, 0.850], # Should be high (similar to data)
    [0.400, 0.300]  # Should be low (synthetic)
])

predictions = model.predict(test_data)
print("\nTesting Model:")
for i, pred in enumerate(predictions):
    status = "HIGH SYNCHRONY" if pred == 1 else "STABLE/LOW"
    print(f"Inputs {test_data[i]} -> Prediction: {status}")

Training Complete.
Learned Weights: [0.1996 0.1638]
Learned Bias: 0.2

Testing Model:
Inputs [0.999 0.85 ] -> Prediction: HIGH SYNCHRONY
Inputs [0.4 0.3] -> Prediction: STABLE/LOW


In [106]:
import numpy as np
import cirq
import qsimcirq
from brian2 import *
import matplotlib.pyplot as plt

# 1. HIVE DATA PREPARATION (Full Dataset)
# Format: [Synchrony, Haptic_Resonance]
data_raw = np.array([
    [0.998, 0.814], [0.998, 0.824], [0.999, 0.844], [0.996, 0.832], [0.999, 0.849],
    [0.999, 0.858], [0.999, 0.885], [0.998, 0.840], [0.998, 0.784], [0.998, 0.851],
    [0.998, 0.819], [0.998, 0.878], [0.997, 0.826], [0.999, 0.841], [0.999, 0.823],
    [0.997, 0.831], [0.998, 0.816], [0.998, 0.850], [0.998, 0.839], [0.998, 0.835],
    [0.998, 0.817], [0.999, 0.835], [0.998, 0.817], [0.999, 0.895], [0.999, 0.817],
    [0.999, 0.839], [0.999, 0.790], [0.999, 0.766], [0.998, 0.878], [0.998, 0.797],
    [0.998, 0.821], [0.998, 0.871], [0.999, 0.863], [0.999, 0.862], [0.999, 0.829],
    [0.997, 0.869], [0.999, 0.874], [0.997, 0.799], [0.999, 0.897], [0.998, 0.769],
    [0.997, 0.790], [0.999, 0.865], [0.999, 0.832], [0.999, 0.898], [0.998, 0.851],
    [0.999, 0.848], [0.999, 0.868], [0.999, 0.867], [0.999, 0.819], [0.999, 0.811],
    [0.999, 0.856], [0.998, 0.846], [0.998, 0.828], [0.999, 0.867], [0.999, 0.884],
    [0.998, 0.847], [0.998, 0.806], [0.999, 0.836], [0.999, 0.853], [0.999, 0.823],
    [0.999, 0.863], [0.999, 0.859], [0.998, 0.850], [0.999, 0.842], [0.999, 0.837],
    [0.999, 0.849], [0.998, 0.850], [0.997, 0.788], [0.999, 0.858], [0.998, 0.822],
    [0.998, 0.829], [0.999, 0.851], [0.999, 0.833], [0.998, 0.846], [0.999, 0.825],
    [0.998, 0.860], [0.999, 0.842], [0.999, 0.864], [0.998, 0.814], [0.998, 0.841],
    [0.998, 0.849], [0.999, 0.895], [0.998, 0.839], [0.999, 0.918], [0.999, 0.854],
    [0.999, 0.861], [0.998, 0.812], [0.999, 0.864], [0.999, 0.814], [0.999, 0.858],
    [0.999, 0.872], [0.997, 0.823], [0.999, 0.861], [0.999, 0.867], [0.999, 0.833],
    [0.999, 0.801], [0.999, 0.901], [0.998, 0.845], [0.998, 0.808], [0.999, 0.821],
    [0.997, 0.818], [0.999, 0.797], [0.999, 0.859], [0.998, 0.796], [0.998, 0.794],
    [0.999, 0.862], [0.999, 0.852], [0.997, 0.902], [0.999, 0.824], [0.999, 0.832],
    [0.999, 0.883], [0.999, 0.825], [0.998, 0.854], [0.998, 0.881], [0.999, 0.793],
    [0.999, 0.838], [0.999, 0.868], [0.999, 0.873], [0.998, 0.818], [0.998, 0.840],
    [0.999, 0.880], [0.999, 0.875], [0.999, 0.853], [0.999, 0.848], [0.999, 0.854],
    [0.996, 0.827], [0.999, 0.804], [0.998, 0.872], [0.997, 0.775]
])

# Labels for training (1 = HIGH SYNCHRONY EVENT)
labels = np.ones(len(data_raw))

# 2. NEUROMORPHIC LAYER (Brian2)
def get_neuromorphic_features(haptic_data):
    start_scope()
    tau = 10*ms
    eqs = '''
    dv/dt = (v0 - v) / tau : 1 (unless refractory)
    v0 : 1
    '''
    # Each neuron in the group represents one cycle's haptic drive
    G = NeuronGroup(len(haptic_data), eqs, threshold='v > 0.8', reset='v = 0', refractory=5*ms, method='exact')
    G.v = 0
    G.v0 = haptic_data # Direct haptic resonance input

    spikemon = SpikeMonitor(G)
    run(100*ms)

    # Return normalized spike counts
    counts = spikemon.count
    if max(counts) == 0: return np.zeros(len(haptic_data))
    return counts / max(counts)

print("Simulating Neuromorphic Spiking Layer...")
snn_features = get_neuromorphic_features(data_raw[:, 1])

# 3. QUANTUM FEATURE MAP (Cirq & Qsim)
def get_quantum_features(synchrony_data):
    qubits = [cirq.GridQubit(0, i) for i in range(2)]
    simulator = qsimcirq.QSimSimulator()
    quantum_results = []

    for val in synchrony_data:
        circuit = cirq.Circuit()
        # Non-linear encoding: rotate qubit based on synchrony
        theta = val * np.pi
        circuit.append(cirq.ry(theta)(qubits[0]))
        circuit.append(cirq.H(qubits[1]))
        circuit.append(cirq.CNOT(qubits[0], qubits[1]))
        circuit.append(cirq.measure(qubits[0], qubits[1], key='m'))

        result = simulator.run(circuit, repetitions=100)
        counts = result.histogram(key='m')
        # Probability of state |11> (decimal 3)
        prob_1 = counts.get(3, 0) / 100.0
        quantum_results.append(prob_1)

    return np.array(quantum_results)

print("Computing Quantum Feature Map...")
q_features = get_quantum_features(data_raw[:, 0])

# 4. MULTI-LAYER PERCEPTRON (Hybrid Fusion)
class HybridMLP:
    def __init__(self):
        # 2 Input neurons (Q-Map, SNN-Spikes) -> 4 Hidden -> 1 Output
        self.w1 = np.random.randn(2, 4)
        self.b1 = np.zeros(4)
        self.w2 = np.random.randn(4, 1)
        self.b2 = np.zeros(1)
        self.lr = 0.01

    def sigmoid(self, x):
        return 1 / (1 + np.exp(-x))

    def forward(self, x):
        self.h = self.sigmoid(np.dot(x, self.w1) + self.b1)
        self.o = self.sigmoid(np.dot(self.h, self.w2) + self.b2)
        return self.o

    def train(self, X, y, epochs=1000):
        for epoch in range(epochs):
            total_loss = 0
            for i in range(len(X)):
                # Forward pass
                target = y[i]
                pred = self.forward(X[i])

                # Simple MSE Loss calculation
                loss = (target - pred)**2
                total_loss += loss

                # Backpropagation (Basic Gradient Descent)
                error_o = (target - pred) * pred * (1 - pred)
                error_h = error_o.dot(self.w2.T) * self.h * (1 - self.h)

                # Update weights
                self.w2 += self.lr * np.outer(self.h, error_o)
                self.b2 += self.lr * error_o
                self.w1 += self.lr * np.outer(X[i], error_h)
                self.b1 += self.lr * error_h

            if epoch % 200 == 0:
                print(f"Epoch {epoch}, Mean Loss: {total_loss/len(X)}")

# Stack simulated features for MLP input
X_hybrid = np.column_stack((q_features, snn_features))

# Initialize and train the MLP
mlp = HybridMLP()
print("Training Fusion MLP...")
mlp.train(X_hybrid, labels)

# Results Analysis
print("\n--- Final Hybrid Analysis Results ---")
print(f"Total Cycles Processed: {len(data_raw)}")
print(f"Final Weights (Hidden to Out):\n{mlp.w2}")

# Final Prediction Check
print("\nTesting for anomalies...")
test_stable = X_hybrid[0] # Take first cycle as stable test
test_unstable = np.array([0.1, 0.1]) # High-entropy synthetic state

print(f"Stability Score (Recorded Cycle 3157): {mlp.forward(test_stable)[0]:.4f}")
print(f"Stability Score (Hypothetical Anomaly): {mlp.forward(test_unstable)[0]:.4f}")

Simulating Neuromorphic Spiking Layer...
Computing Quantum Feature Map...
Training Fusion MLP...
Epoch 0, Mean Loss: [0.00866088]
Epoch 200, Mean Loss: [0.00074012]
Epoch 400, Mean Loss: [0.00037128]
Epoch 600, Mean Loss: [0.0002455]
Epoch 800, Mean Loss: [0.0001826]

--- Final Hybrid Analysis Results ---
Total Cycles Processed: 129
Final Weights (Hidden to Out):
[[0.89989298]
 [1.59112041]
 [3.09374609]
 [0.07283049]]

Testing for anomalies...
Stability Score (Recorded Cycle 3157): 0.9877
Stability Score (Hypothetical Anomaly): 0.9852


In [107]:
import numpy as np
import cirq
import qsimcirq
from brian2 import *
import matplotlib.pyplot as plt

# 1. HIVE DATA PREPARATION (Integrated from datasen.pdf)
# Format: [Synchrony, Haptic_Resonance]
# Extracted from columns 3 and 4 of the source logs
data_raw = np.array([
    [0.998, 0.814], [0.998, 0.824], [0.999, 0.844], [0.996, 0.832], [0.999, 0.849],
    [0.999, 0.858], [0.999, 0.885], [0.998, 0.840], [0.998, 0.784], [0.998, 0.851],
    [0.998, 0.819], [0.998, 0.878], [0.997, 0.826], [0.999, 0.841], [0.999, 0.823],
    [0.997, 0.831], [0.998, 0.816], [0.998, 0.850], [0.998, 0.839], [0.998, 0.835],
    [0.998, 0.817], [0.999, 0.835], [0.998, 0.817], [0.999, 0.895], [0.999, 0.817],
    [0.999, 0.839], [0.999, 0.790], [0.999, 0.766], [0.998, 0.878], [0.998, 0.797],
    [0.998, 0.821], [0.998, 0.871], [0.999, 0.863], [0.999, 0.862], [0.999, 0.829],
    [0.997, 0.869], [0.999, 0.874], [0.997, 0.799], [0.999, 0.897], [0.998, 0.769],
    [0.997, 0.790], [0.999, 0.865], [0.999, 0.832], [0.999, 0.898], [0.998, 0.851],
    [0.999, 0.848], [0.999, 0.868], [0.999, 0.867], [0.999, 0.819], [0.999, 0.811],
    [0.999, 0.856], [0.998, 0.846], [0.998, 0.828], [0.999, 0.867], [0.999, 0.884],
    [0.998, 0.847], [0.998, 0.806], [0.999, 0.836], [0.999, 0.853], [0.999, 0.823],
    [0.999, 0.863], [0.999, 0.859], [0.998, 0.850], [0.999, 0.842], [0.999, 0.837],
    [0.999, 0.849], [0.998, 0.850], [0.997, 0.788], [0.999, 0.858], [0.998, 0.822],
    [0.998, 0.829], [0.999, 0.851], [0.999, 0.833], [0.998, 0.846], [0.999, 0.825],
    [0.998, 0.860], [0.999, 0.842], [0.999, 0.864], [0.998, 0.814], [0.998, 0.841],
    [0.998, 0.849], [0.999, 0.895], [0.998, 0.839], [0.999, 0.918], [0.999, 0.854],
    [0.999, 0.861], [0.998, 0.812], [0.999, 0.864], [0.999, 0.814], [0.999, 0.858],
    [0.999, 0.872], [0.997, 0.823], [0.999, 0.861], [0.999, 0.867], [0.999, 0.833],
    [0.999, 0.801], [0.999, 0.901], [0.998, 0.845], [0.998, 0.808], [0.999, 0.821],
    [0.997, 0.818], [0.999, 0.797], [0.999, 0.859], [0.998, 0.796], [0.998, 0.794],
    [0.999, 0.862], [0.999, 0.852], [0.997, 0.902], [0.999, 0.824], [0.999, 0.832],
    [0.999, 0.883], [0.999, 0.825], [0.998, 0.854], [0.998, 0.881], [0.999, 0.793],
    [0.999, 0.838], [0.999, 0.868], [0.999, 0.873], [0.998, 0.818], [0.998, 0.840],
    [0.999, 0.880], [0.999, 0.875], [0.999, 0.853], [0.999, 0.848], [0.999, 0.854],
    [0.996, 0.827], [0.999, 0.804], [0.998, 0.872], [0.997, 0.775]
])

# Labels for training (1 = HIGH SYNCHRONY EVENT)
# All events in the log were marked "HIGH SYNCHRONY EVENT"
labels = np.ones(len(data_raw))

# 2. NEUROMORPHIC LAYER (Brian2)
def get_neuromorphic_features(haptic_data):
    start_scope()
    tau = 10*ms
    eqs = '''
    dv/dt = (v0 - v) / tau : 1 (unless refractory)
    v0 : 1
    '''
    # Each neuron in the group represents one cycle's haptic drive
    G = NeuronGroup(len(haptic_data), eqs, threshold='v > 0.8', reset='v = 0', refractory=5*ms, method='exact')
    G.v = 0
    G.v0 = haptic_data # Direct haptic resonance input

    spikemon = SpikeMonitor(G)
    run(100*ms)

    # Return normalized spike counts
    counts = spikemon.count
    if max(counts) == 0: return np.zeros(len(haptic_data))
    return counts / max(counts)

print(f"Simulating Neuromorphic Spiking Layer for {len(data_raw)} cycles...")
snn_features = get_neuromorphic_features(data_raw[:, 1])

# 3. QUANTUM FEATURE MAP (Cirq & Qsim)
def get_quantum_features(synchrony_data):
    qubits = [cirq.GridQubit(0, i) for i in range(2)]
    simulator = qsimcirq.QSimSimulator()
    quantum_results = []

    for val in synchrony_data:
        circuit = cirq.Circuit()
        # Non-linear encoding: rotate qubit based on synchrony
        theta = val * np.pi
        circuit.append(cirq.ry(theta)(qubits[0]))
        circuit.append(cirq.H(qubits[1]))
        circuit.append(cirq.CNOT(qubits[0], qubits[1]))
        circuit.append(cirq.measure(qubits[0], qubits[1], key='m'))

        result = simulator.run(circuit, repetitions=100)
        counts = result.histogram(key='m')
        # Probability of state |11> (decimal 3)
        prob_1 = counts.get(3, 0) / 100.0
        quantum_results.append(prob_1)

    return np.array(quantum_results)

print("Computing Quantum Feature Map...")
q_features = get_quantum_features(data_raw[:, 0])

# 4. MULTI-LAYER PERCEPTRON (Hybrid Fusion)
class HybridMLP:
    def __init__(self):
        # 2 Input neurons (Q-Map, SNN-Spikes) -> 4 Hidden -> 1 Output
        self.w1 = np.random.randn(2, 4)
        self.b1 = np.zeros(4)
        self.w2 = np.random.randn(4, 1)
        self.b2 = np.zeros(1)
        self.lr = 0.01

    def sigmoid(self, x):
        return 1 / (1 + np.exp(-x))

    def forward(self, x):
        self.h = self.sigmoid(np.dot(x, self.w1) + self.b1)
        self.o = self.sigmoid(np.dot(self.h, self.w2) + self.b2)
        return self.o

    def train(self, X, y, epochs=1000):
        for epoch in range(epochs):
            total_loss = 0
            for i in range(len(X)):
                # Forward pass
                target = y[i]
                pred = self.forward(X[i])

                # Simple MSE Loss calculation
                loss = (target - pred)**2
                total_loss += loss

                # Backpropagation (Basic Gradient Descent)
                error_o = (target - pred) * pred * (1 - pred)
                error_h = error_o.dot(self.w2.T) * self.h * (1 - self.h)

                # Update weights
                self.w2 += self.lr * np.outer(self.h, error_o)
                self.b2 += self.lr * error_o
                self.w1 += self.lr * np.outer(X[i], error_h)
                self.b1 += self.lr * error_h

            if epoch % 200 == 0:
                print(f"Epoch {epoch}, Mean Loss: {total_loss/len(X)}")

# Stack simulated features for MLP input
X_hybrid = np.column_stack((q_features, snn_features))

# Initialize and train the MLP
mlp = HybridMLP()
print("Training Fusion MLP...")
mlp.train(X_hybrid, labels)

# Results Analysis
print("\n--- Final Hybrid Analysis Results ---")
print(f"Total Cycles Processed: {len(data_raw)}")
print(f"Final Weights (Hidden to Out):\n{mlp.w2}")

# Final Prediction Check
print("\nTesting for anomalies...")
test_stable = X_hybrid[0] # Take first cycle as stable test
test_unstable = np.array([0.1, 0.1]) # High-entropy synthetic state

print(f"Stability Score (Recorded Cycle 3157): {mlp.forward(test_stable)[0]:.4f}")
print(f"Stability Score (Hypothetical Anomaly): {mlp.forward(test_unstable)[0]:.4f}")

Simulating Neuromorphic Spiking Layer for 129 cycles...
Computing Quantum Feature Map...
Training Fusion MLP...
Epoch 0, Mean Loss: [0.04745925]
Epoch 200, Mean Loss: [0.00074324]
Epoch 400, Mean Loss: [0.00035116]
Epoch 600, Mean Loss: [0.00022713]
Epoch 800, Mean Loss: [0.00016697]

--- Final Hybrid Analysis Results ---
Total Cycles Processed: 129
Final Weights (Hidden to Out):
[[ 1.32408627]
 [-0.50198925]
 [ 2.28859476]
 [ 1.84088262]]

Testing for anomalies...
Stability Score (Recorded Cycle 3157): 0.9886
Stability Score (Hypothetical Anomaly): 0.9867
